<a href="https://colab.research.google.com/github/pgrady1322/T2T-Polish/blob/main/ErrorSmith_Training_Colab_Mar9_Overnight.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔬 StrandWeaver — ErrorSmith Training (Colab GPU)

Train a **per-chemistry ensemble** of XGBoost classifiers for per-base sequencing error prediction.
Each chemistry family gets its own dedicated 5-class model.
Training uses **chemistry-specific** (flow cell / machine / chemistry) BAMs aligned to **HG002v1.0.1** (Q100 ground truth) and/or **CHM13v2.0**.

| Class | Label | Description |
|-------|-------|-------------|
| 0 | `correct` | Base matches reference |
| 1 | `substitution` | Wrong base |
| 2 | `insertion` | Extra base in read |
| 3 | `deletion` | Missing base in read |
| 4 | `homopolymer_error` | Ins/del in homopolymer context |

**Data sources:**
- **HG002v1.0.1** — Q100 T2T assembly, 10 BAMs across 10 chemistry codes (see config cell)
- **CHM13v2.0** — Original training reference (legacy, 6 chemistries)

| Code | Chemistry | Platform | HG002 BAMs |
|------|-----------|----------|------------|
| 0 | `pacbio_hifi_sequel2` | PacBio HiFi (Sequel II/IIe) | — |
| 1 | `ont_lsk110_r941` | ONT LSK-110, R9.4.1 | — |
| 2 | `ont_ulk001_r941` | ONT ULK-001, R9.4.1 (ultra long) | — |
| 3 | `ont_lsk114_r1041` | ONT LSK-114, R10.4.1 (standard ligation) | — |
| 4 | `ont_ulk114_r1041` | ONT ULK-114, R10.4.1 (standard ultra long) | ✓ |
| 5 | `illumina_hiseq2500` | Illumina HiSeq 2500 | ✓ 2×250 |
| 6 | `pacbio_onso` | PacBio Onso (short-read SBB, NOT HiFi) | ✓ NIST 2024Q1 |
| 7 | `element_aviti` | Element Aviti (standard / long) | ✓ Standard + Long |
| 8 | `element_ultraq` | Element UltraQ | ✓ NIST Jun 2024 |
| 9 | `pacbio_hifi_revio` | PacBio HiFi (Revio) | ✓ 3-cell |
| 10 | `ont_r1041_duplex` | ONT R10.4.1 Duplex | ✓ |
| 11 | `ont_ulk114_r1041_hiacc` | ONT UL R10.4.1 High-Accuracy (exp.) | ✓ |
| 12 | `ont_ulk114_r1041_dorado` | ONT UL R10.4.1 Dorado | ✓ |

### Setup
1. `Runtime → Change runtime type → T4 GPU` (or A100 if available)
2. Run cells top-to-bottom

### Data options
- **Option 1 (recommended):** Transfer HG002 BAMs to Google Drive via E2 rclone (see `Non_Main_Commit_Files/hg002_transfer/`)
- **Option 2:** Upload pre-aligned CHM13 BAMs to Google Drive `ErrorSmith_bams/`

> **K-Weaver training** is in a separate notebook: `KWeaver_Training_Colab.ipynb`

## 1. Environment Setup

In [1]:
# ── Verify GPU ──────────────────────────────────────────────────────
import subprocess, sys, os

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print(f"✓ GPU detected: {result.stdout.strip()}")
    GPU_AVAILABLE = True
else:
    print("⚠ No GPU detected — XGBoost will use CPU (still fast, just slower HP search)")
    GPU_AVAILABLE = False

# Check RAM
import psutil
ram_gb = psutil.virtual_memory().total / 1e9
print(f"✓ RAM: {ram_gb:.1f} GB {'(High-RAM ✓)' if ram_gb > 20 else '(consider High-RAM runtime)'}")
print(f"✓ Python: {sys.version.split()[0]}")

✓ GPU detected: NVIDIA L4, 23034 MiB, 580.82.07
✓ RAM: 56.9 GB (High-RAM ✓)
✓ Python: 3.12.12


In [10]:
# ── Install dependencies ────────────────────────────────────────────
!pip install -q xgboost scikit-learn numpy pandas optuna pysam biopython

# Clone StrandWeaver (dev branch) and install — pull latest on re-runs
!git clone -b dev https://github.com/pgrady1322/strandweaver.git /content/strandweaver 2>/dev/null || \
    (cd /content/strandweaver && git pull origin dev)
!cd /content/strandweaver && pip install -e . -q

print("\n✓ Dependencies installed")

From https://github.com/pgrady1322/strandweaver
 * branch            dev        -> FETCH_HEAD
Already up to date.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for strandweaver (pyproject.toml) ... done

✓ Dependencies installed


In [3]:
# ── Mount Google Drive (for saving results) ─────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Directory layout ────────────────────────────────────────────────
GDRIVE_DIR        = '/content/drive/MyDrive/Colab_Training'
ERRORSMITH_OUT    = '/content/errorsmith_training'
MODELS_OUT        = '/content/trained_models'
GDRIVE_ERRORSMITH = os.path.join(GDRIVE_DIR, 'errorsmith_models')

for d in [GDRIVE_DIR, ERRORSMITH_OUT, MODELS_OUT]:
    os.makedirs(d, exist_ok=True)

print(f"✓ Google Drive mounted")
print(f"✓ Working dirs created")
print(f"  ErrorSmith output:  {ERRORSMITH_OUT}")
print(f"  Models output:      {MODELS_OUT}")
print(f"  Drive export:       {GDRIVE_DIR}")

Mounted at /content/drive
✓ Google Drive mounted
✓ Working dirs created
  ErrorSmith output:  /content/errorsmith_training
  Models output:      /content/trained_models
  Drive export:       /content/drive/MyDrive/Colab_Training


In [4]:
# ── Verify StrandWeaver imports ──────────────────────────────────────
sys.path.insert(0, '/content/strandweaver')
sys.path.insert(0, '/content/strandweaver/trained_models/training/scripts/')

from strandweaver.io_utils import SeqRead

import xgboost as xgb
import numpy as np
import pandas as pd
import time
import logging
from pathlib import Path

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s', datefmt='%H:%M:%S')
logger = logging.getLogger('colab_training')

SEED = 42

# XGBoost device config
XGB_DEVICE = 'cuda' if GPU_AVAILABLE else 'cpu'
XGB_TREE = 'hist'  # gpu_hist is deprecated in XGBoost 2.x — 'hist' auto-uses GPU

print(f"✓ StrandWeaver imports OK")
print(f"✓ XGBoost {xgb.__version__}")
print(f"✓ NumPy {np.__version__}")
print(f"✓ Pandas {pd.__version__}")

✓ StrandWeaver imports OK
✓ XGBoost 3.2.0
✓ NumPy 2.0.2
✓ Pandas 2.2.2


## 2. Configuration & Data Source

**Edit the cell below to match your setup.**

### Option 1: HG002v1.0.1 BAMs from Google Drive (recommended)

Transfer HG002 BAMs using the E2 rclone scripts in `Non_Main_Commit_Files/hg002_transfer/`.
After transfer, BAMs will be at:
```
My Drive/Colab_Training/ErrorSmith_bams/HG002/
├── illumina/             (HiSeq 2500, 2×250)           code 5
├── pacbio_hifi_revio/    (Revio HiFi)                  code 9
├── pacbio_onso/          (Onso short-read — NOT HiFi)   code 6
├── ont_lsk114_duplex/    (R10.4.1 Duplex)               code 10
├── ont_ulk114/           (UL R10 standard)              code 4
├── ont_ulk114_hiacc/     (UL R10 HiAcc experimental)    code 11
├── ont_ulk114_dorado/    (UL R10 Dorado basecaller)     code 12
├── element_aviti/        (Aviti Standard + Long)         code 7
├── element_ultraq/       (UltraQ)                       code 8
└── reference/            (HG002v1.0.1.fasta.gz)
```

### Option 2: CHM13 BAMs (legacy)

If using CHM13 data, upload to `ErrorSmith_bams/CHM13/` — the config cell detects both.

### Option 3: Upload FASTQs → align on Colab

Set `ERRORSMITH_MODE = 'fastq'` and provide FASTQs in `Colab_Training/fastq/`.

In [5]:
# ══════════════════════════════════════════════════════════════════════
#  ERRORSMITH CONFIGURATION — edit these before running
# ══════════════════════════════════════════════════════════════════════

# ── Data source mode ────────────────────────────────────────────────
# 'bam'  : Use pre-aligned BAMs on Google Drive (recommended)
# 'fastq': Upload FASTQs to Google Drive, align on Colab
ERRORSMITH_MODE = 'bam'

# ── BAM directory (all BAMs live here) ──────────────────────────────
BAM_DIR = os.path.join(GDRIVE_DIR, 'ErrorSmith_bams')
HG002_BAM_DIR = os.path.join(BAM_DIR, 'HG002')
CHM13_BAM_DIR = os.path.join(BAM_DIR, 'CHM13')

# ══════════════════════════════════════════════════════════════════════
#  HG002v1.0.1 BAM MAP (Q100 ground truth — primary training data)
# ══════════════════════════════════════════════════════════════════════
# Transferred via Non_Main_Commit_Files/hg002_transfer/transfer_hg002_bams.sh
# Each chemistry key matches a CHEMISTRY_CODES entry (0–13).

HG002_BAM_MAP = {
    # Code 4: ONT ULK-114 R10.4.1 (ultra-long, standard basecaller)
    'ont_ulk114_r1041': [
        os.path.join(HG002_BAM_DIR, 'ont_ulk114', 'hg002v1.0_ont_r10_ul.pri.bam'),
    ],
    # Code 5: Illumina HiSeq 2500 (2×250)
    'illumina_hiseq2500': [
        os.path.join(HG002_BAM_DIR, 'illumina', 'hg002v1.0.1_illumina2x250_hg002.dedup.bam'),
    ],
    # Code 6: PacBio Onso (short-read SBB — NOT HiFi, entirely different)
    'pacbio_onso': [
        os.path.join(HG002_BAM_DIR, 'pacbio_onso', 'hg002v1.0.1_onso_hg002_NIST_2024Q1.dedup.bam'),
    ],
    # Code 7: Element Aviti Standard (2×150 bp)
    'element_aviti': [
        os.path.join(HG002_BAM_DIR, 'element_aviti', 'hg002v1.0.1_element_hg002_avitistd.dedup.bam'),
    ],
    # Code 13: Element Aviti Long (2×300 bp)
    'element_aviti_lng': [
        os.path.join(HG002_BAM_DIR, 'element_aviti', 'hg002v1.0.1_hg002_element_avitilng.dedup.bam'),
    ],
    # Code 8: Element UltraQ
    'element_ultraq': [
        os.path.join(HG002_BAM_DIR, 'element_ultraq', 'hg002v1.0.1_element_hg002_ultraq_jun2024.1.dedup.bam'),
    ],
    # Code 9: PacBio HiFi Revio (distinct from Sequel II)
    'pacbio_hifi_revio': [
        os.path.join(HG002_BAM_DIR, 'pacbio_hifi_revio', 'hg002v1.0_hifi_revio_3cells.pri.bam'),
    ],
    # Code 10: ONT R10.4.1 Duplex basecalling
    'ont_r1041_duplex': [
        os.path.join(HG002_BAM_DIR, 'ont_lsk114_duplex', 'hg002v1.0_ont_r10_duplex.pri.bam'),
    ],
    # Code 11: ONT UL R10.4.1 High-Accuracy (experimental)
    'ont_ulk114_r1041_hiacc': [
        os.path.join(HG002_BAM_DIR, 'ont_ulk114_hiacc', 'all_pass.vhg002v1.bam'),
    ],
    # Code 12: ONT UL R10.4.1 Dorado basecaller
    'ont_ulk114_r1041_dorado': [
        os.path.join(HG002_BAM_DIR, 'ont_ulk114_dorado', 'hg002v1.0_ont_r10_ul_dorado.pri.bam'),
    ],
}

# ══════════════════════════════════════════════════════════════════════
#  CHM13 BAM MAP (processed against CHM13v2.0 reference)
# ══════════════════════════════════════════════════════════════════════
CHM13_BAM_MAP = {
    'pacbio_hifi_sequel2': [
        os.path.join(CHM13_BAM_DIR, 'chm13_SRR11292120_HiFi.sorted.bam'),
        os.path.join(CHM13_BAM_DIR, 'chm13_SRR11292121_HiFi.sorted.bam'),
    ],
    'ont_lsk110_r941': [
        os.path.join(CHM13_BAM_DIR, 'chm13_SRR31588525_ONT_SL_R941.sorted.bam'),
    ],
    'ont_ulk001_r941': [
        os.path.join(CHM13_BAM_DIR, 'chm13_SRR23513621_ONT_UL_R941.sorted.bam'),
    ],
    'ont_lsk114_r1041': [
        os.path.join(CHM13_BAM_DIR, 'chm13_SRR35645112_ONT_SL_R1041.sorted.bam'),
    ],
    'ont_ulk114_r1041':  [
        os.path.join(CHM13_BAM_DIR, 'chm13_SRR34901565_ONT_UL_R1041.sorted.bam'),
    ],
    'illumina_hiseq2500': [
        os.path.join(CHM13_BAM_DIR, 'chm13_SRR1997411_Illumina_PCR_Free.sorted.bam'),
    ],
}

# ── FASTQ paths (for 'fastq' mode only) ────────────────────────────
FASTQ_DIR = os.path.join(GDRIVE_DIR, 'fastq')

FASTQ_MAP = {
    'pacbio_hifi_sequel2': [
        os.path.join(FASTQ_DIR, 'SRR11292120.fastq.gz'),
        os.path.join(FASTQ_DIR, 'SRR11292121.fastq.gz'),
    ],
    'ont_lsk110_r941':     [],
    'ont_ulk001_r941': [
        os.path.join(FASTQ_DIR, 'chm13_ont_ul.fastq.gz'),
    ],
    'ont_lsk114_r1041':            [],
    'ont_r1041_duplex':            [],
    'ont_ulk114_r1041':            [],
    'ont_ulk114_r1041_hiacc':      [],
    'ont_ulk114_r1041_dorado':     [],
    'illumina_hiseq2500': [
        os.path.join(FASTQ_DIR, 'SRR1997411_1.fastq.gz'),
        os.path.join(FASTQ_DIR, 'SRR1997411_2.fastq.gz'),
    ],
    'element_aviti_lng':   [],     # HG002 BAMs already aligned
    'element_ultraq':      [],     # HG002 BAMs already aligned
}

# ── Reference genomes (dual-reference: each BAM set uses its own) ──
HG002_REF_PATH = os.path.join(GDRIVE_DIR, 'ErrorSmith_bams', 'HG002', 'reference', 'hg002v1.0.1.fasta')
CHM13_REF_PATH = os.path.join(GDRIVE_DIR, 'chm13v2.0.fa')

AUTO_DOWNLOAD_REF = True

# ── Subsampling ─────────────────────────────────────────────────────
ERRORSMITH_SUBSAMPLE = 10000000  # None = no limit, use ALL data

# ── Chromosomes to process ──────────────────────────────────────────
ERRORSMITH_CHROMS = None  # None = all chromosomes (auto-detected from BAM ∩ reference)

# ── Threads ─────────────────────────────────────────────────────────
ERRORSMITH_THREADS = 32

# ── Validate inputs ────────────────────────────────────────────────
from generate_errorsmith_training_data import CHEMISTRY_CODES, CHEMISTRY_PRESETS

print(f"Mode: {ERRORSMITH_MODE}")
print(f"Dual-reference: HG002 BAMs → HG002v1.0.1 | CHM13 BAMs → CHM13v2.0\n")

# Check references
for label, ref in [('HG002v1.0.1', HG002_REF_PATH), ('CHM13v2.0', CHM13_REF_PATH)]:
    gz = ref + '.gz'
    if os.path.exists(ref):
        print(f"  ✓ {label}: {ref}")
    elif os.path.exists(gz):
        print(f"  ✓ {label}: {gz}")
    else:
        print(f"  ✗ {label}: not found — {ref}")

if ERRORSMITH_MODE == 'bam':
    print("\nHG002 BAMs (→ HG002v1.0.1 reference):")
    for chem, paths in HG002_BAM_MAP.items():
        found = [p for p in paths if os.path.exists(p)]
        missing = [p for p in paths if not os.path.exists(p)]
        if found:
            print(f"  ✓ {chem:30s}: {len(found)} BAM(s) found")
            for f in found:
                sz = os.path.getsize(f) / (1024**3)
                print(f"      {os.path.basename(f)} ({sz:.1f} GB)")
        if missing:
            for f in missing:
                print(f"  ✗ {chem:30s}: not found — {os.path.basename(f)}")

    print("\nCHM13 BAMs (→ CHM13v2.0 reference):")
    for chem, paths in CHM13_BAM_MAP.items():
        found = [p for p in paths if os.path.exists(p)]
        missing = [p for p in paths if not os.path.exists(p)]
        if found:
            print(f"  ✓ {chem:30s}: {len(found)} BAM(s) found")
            for f in found:
                sz = os.path.getsize(f) / (1024**3)
                print(f"      {os.path.basename(f)} ({sz:.1f} GB)")
        if missing:
            for f in missing:
                print(f"  ✗ {chem:30s}: not found — {os.path.basename(f)}")

elif ERRORSMITH_MODE == 'fastq':
    print("\nChemistry-specific FASTQs:")
    for chem in CHEMISTRY_CODES:
        fqs = FASTQ_MAP.get(chem, [])
        if not fqs:
            print(f"  ⏳ {chem:30s}: no FASTQs (skipping)")
            continue
        found = [f for f in fqs if os.path.exists(f)]
        missing = [f for f in fqs if not os.path.exists(f)]
        if found:
            print(f"  ✓ {chem:30s}: {len(found)} file(s) found")
        if missing:
            for f in missing:
                print(f"  ✗ {chem:30s}: not found — {os.path.basename(f)}")

sub_str = f"{ERRORSMITH_SUBSAMPLE:,} bases/chemistry" if ERRORSMITH_SUBSAMPLE else "None (all data)"
chr_str = str(ERRORSMITH_CHROMS) if ERRORSMITH_CHROMS else "all (auto-detect from BAM ∩ reference)"
print(f"\n  Subsample:    {sub_str}")

print(f"  Chromosomes:  {chr_str}")


Mode: bam
Dual-reference: HG002 BAMs → HG002v1.0.1 | CHM13 BAMs → CHM13v2.0

  ✓ HG002v1.0.1: /content/drive/MyDrive/Colab_Training/ErrorSmith_bams/HG002/reference/hg002v1.0.1.fasta.gz
  ✓ CHM13v2.0: /content/drive/MyDrive/Colab_Training/chm13v2.0.fa.gz

HG002 BAMs (→ HG002v1.0.1 reference):
  ✗ ont_ulk114_r1041              : not found — hg002v1.0_ont_r10_ul.pri.bam
  ✓ illumina_hiseq2500            : 1 BAM(s) found
      hg002v1.0.1_illumina2x250_hg002.dedup.bam (128.4 GB)
  ✓ pacbio_onso                   : 1 BAM(s) found
      hg002v1.0.1_onso_hg002_NIST_2024Q1.dedup.bam (90.1 GB)
  ✓ element_aviti                 : 1 BAM(s) found
      hg002v1.0.1_element_hg002_avitistd.dedup.bam (126.8 GB)
  ✓ element_aviti_lng             : 1 BAM(s) found
      hg002v1.0.1_hg002_element_avitilng.dedup.bam (104.5 GB)
  ✓ element_ultraq                : 1 BAM(s) found
      hg002v1.0.1_element_hg002_ultraq_jun2024.1.dedup.bam (131.0 GB)
  ✓ pacbio_hifi_revio             : 1 BAM(s) found
      hg00

## 3. Download Reference + Install Tools + Align FASTQs

In [6]:
# ── Download references if needed ────────────────────────────────────
from generate_errorsmith_training_data import download_reference, CHEMISTRY_PRESETS

# HG002v1.0.1 reference
hg002_ref_exists = os.path.exists(HG002_REF_PATH) or os.path.exists(HG002_REF_PATH + '.gz')
if hg002_ref_exists:
    print(f"✓ HG002v1.0.1 reference found")
else:
    print(f"⚠ HG002v1.0.1 reference not found at {HG002_REF_PATH}")
    print(f"  HG002 BAMs will be skipped unless the reference is provided.")

# CHM13v2.0 reference (auto-download available)
chm13_ref_exists = os.path.exists(CHM13_REF_PATH) or os.path.exists(CHM13_REF_PATH + '.gz')
if not chm13_ref_exists and AUTO_DOWNLOAD_REF:
    print("Downloading CHM13v2.0 reference (800 MB)...")
    ref_dir = os.path.dirname(CHM13_REF_PATH)
    downloaded_ref = download_reference(Path(ref_dir))
    CHM13_REF_PATH = str(downloaded_ref)
    print(f"✓ CHM13v2.0 reference downloaded: {CHM13_REF_PATH}")
elif chm13_ref_exists:
    print(f"✓ CHM13v2.0 reference found")
else:
    print(f"⚠ CHM13v2.0 reference not found at {CHM13_REF_PATH}")
    print(f"  Set AUTO_DOWNLOAD_REF = True to download automatically")

# ── Install minimap2 + samtools (always needed for 'fastq' mode) ────
if ERRORSMITH_MODE == 'fastq':
    print("\nInstalling minimap2, samtools...")
    !apt-get install -qq -y samtools minimap2
    !which minimap2 && echo "✓ minimap2" || echo "✗ minimap2 missing"
    !which samtools && echo "✓ samtools" || echo "✗ samtools missing"

# ── Align FASTQs → sorted BAMs (for 'fastq' mode) ──────────────────
if ERRORSMITH_MODE == 'fastq':
    import subprocess

    BAM_OUT_DIR = os.path.join(ERRORSMITH_OUT, 'bams')
    os.makedirs(BAM_OUT_DIR, exist_ok=True)

    def align_fastqs(fastq_files, chemistry, ref_path, out_dir, threads=4):
        """Align FASTQ(s) with minimap2, sort + index with samtools."""
        preset = CHEMISTRY_PRESETS[chemistry]
        bam_path = os.path.join(out_dir, f'{chemistry}.sorted.bam')
        bai_path = bam_path + '.bai'

        if os.path.exists(bam_path) and os.path.exists(bai_path):
            print(f"  ✓ {chemistry} BAM already exists: {bam_path}")
            return bam_path

        existing = [f for f in fastq_files if os.path.exists(f)]
        if not existing:
            print(f"  ✗ {chemistry}: no FASTQ files found, skipping")
            return None

        print(f"  Aligning {chemistry} ({len(existing)} file(s)) with minimap2 -x {preset}...")
        mm_cmd = ['minimap2', '-ax', preset, '-t', str(threads), ref_path] + existing
        sort_cmd = ['samtools', 'sort', '-@', str(threads), '-o', bam_path]

        with subprocess.Popen(mm_cmd, stdout=subprocess.PIPE) as mm:
            subprocess.run(sort_cmd, stdin=mm.stdout, check=True)

        subprocess.run(['samtools', 'index', '-@', str(threads), bam_path], check=True)
        sz = os.path.getsize(bam_path) / (1024**3)
        print(f"  ✓ {chemistry} BAM ready: {bam_path} ({sz:.1f} GB)")
        return bam_path

    # Align FASTQs — CHM13 BAMs use CHM13 ref, HG002 BAMs use HG002 ref
    # (FASTQ mode is legacy and only supports CHM13 alignment for now)
    chm13_ref = CHM13_REF_PATH if os.path.exists(CHM13_REF_PATH) else CHM13_REF_PATH + '.gz'
    print("\n── Aligning FASTQs to CHM13v2.0 ──────────────────────────")
    for chem, fqs in FASTQ_MAP.items():
        if not fqs:
            print(f"  ⏳ {chem}: no FASTQs, skipping")
            continue
        result = align_fastqs(fqs, chem, chm13_ref, BAM_OUT_DIR, ERRORSMITH_THREADS)
        if result:
            CHM13_BAM_MAP[chem] = [result]

✓ HG002v1.0.1 reference found
✓ CHM13v2.0 reference found


## 4. Generate ErrorSmith Training Data
Processes BAMs **one chemistry at a time**, parses CIGAR alignments, extracts per-base error features.

**Resumability:** Each per-chemistry CSV is checkpointed to Google Drive immediately after generation. On re-run, already-completed chemistries are **skipped automatically**. If Colab crashes mid-run, just re-execute this cell — it picks up where it left off.

Set `FORCE_REGENERATE = True` below to re-process all chemistries from scratch.

In [7]:
# ── Generate ErrorSmith training CSVs (resumable, per-chemistry) ──────
# Each chemistry is processed individually and checkpointed to GDrive.
# On re-run, completed chemistries are skipped automatically.
# At the end, all per-chemistry CSVs are merged into one training CSV.
#
# Checkpoint-aware: even if a BAM has been deleted, the existing
# checkpoint CSV on GDrive will still be picked up for the final merge.
#
# Streaming merge: CSVs are concatenated line-by-line to avoid loading
# the entire dataset into RAM (which can OOM on Colab).
#
# The merged CSV is also written to GDrive. On re-run, if the merged
# file already exists, the entire generation + merge is skipped.

import shutil
import csv as _csv
import traceback
import pandas as pd
from generate_errorsmith_training_data import generate_errorsmith_training_data

# ── Resumability config ─────────────────────────────────────────────
FORCE_REGENERATE = False          # Set True to re-process ALL chemistries
GDRIVE_CHECKPOINT_DIR = os.path.join(GDRIVE_DIR, 'errorsmith_checkpoints')
GDRIVE_MERGED_CSV = os.path.join(GDRIVE_CHECKPOINT_DIR, 'errorsmith_training_merged.csv')
os.makedirs(GDRIVE_CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoint dir: {GDRIVE_CHECKPOINT_DIR}")

def _resolve_bams(bam_map_raw):
    """Flatten per-chemistry BAM lists → resolved_bam_map."""
    resolved = {}
    for chem, paths in bam_map_raw.items():
        if paths is None:
            continue
        for p in paths:
            if os.path.exists(p):
                key = chem
                if key in resolved:
                    i = 2
                    while f"{chem}_part{i}" in resolved:
                        i += 1
                    key = f"{chem}_part{i}"
                resolved[key] = p
            else:
                print(f"  ✗ {chem}: BAM not found — {os.path.basename(p)}")
    return resolved

def _all_declared_keys(bam_map_raw):
    """Return every (chem_key) that would be produced by _resolve_bams,
    including _part2 etc. for multi-BAM entries — even if BAM is missing."""
    keys = []
    seen = set()
    for chem, paths in bam_map_raw.items():
        if paths is None:
            continue
        for _ in paths:
            key = chem
            if key in seen:
                i = 2
                while f"{chem}_part{i}" in seen:
                    i += 1
                key = f"{chem}_part{i}"
            seen.add(key)
            keys.append(key)
    return keys

def _checkpoint_name(ref_label, chem_key):
    """Deterministic checkpoint filename for a (reference, chemistry) pair."""
    return f"{ref_label}_{chem_key}.csv"

def _is_checkpoint_valid(ckpt_path):
    """Check if a checkpoint CSV exists and has data rows."""
    if not os.path.exists(ckpt_path):
        return False
    try:
        sz = os.path.getsize(ckpt_path)
        if sz < 200:  # header-only or corrupt
            return False
        # Quick row count check (header + at least 1 data row)
        with open(ckpt_path, 'r') as f:
            lines = 0
            for _ in f:
                lines += 1
                if lines >= 2:
                    return True
        return False
    except Exception:
        return False

def _count_csv_rows(path):
    """Count data rows in a CSV (excludes header). Memory-efficient."""
    count = -1  # start at -1 to exclude header
    with open(path, 'r') as f:
        for _ in f:
            count += 1
    return max(count, 0)

# ── Check for existing merged CSV on GDrive ─────────────────────────
final_csv = Path(ERRORSMITH_OUT) / 'errorsmith_training.csv'

if not FORCE_REGENERATE and _is_checkpoint_valid(GDRIVE_MERGED_CSV):
    row_count = _count_csv_rows(GDRIVE_MERGED_CSV)
    sz_gb = os.path.getsize(GDRIVE_MERGED_CSV) / (1024**3)
    print(f"✓ Merged CSV already exists on GDrive ({row_count:,} rows, {sz_gb:.2f} GB)")
    print(f"  {GDRIVE_MERGED_CSV}")
    print(f"  Copying to local workspace...")
    shutil.copy2(GDRIVE_MERGED_CSV, final_csv)
    print(f"  ✓ Copied → {final_csv}")
    print(f"\n  Set FORCE_REGENERATE = True to rebuild from scratch.")
    errorsmith_csv = final_csv

else:
    # ── Resolve BAM maps ────────────────────────────────────────────
    hg002_resolved = _resolve_bams(HG002_BAM_MAP)
    chm13_resolved = _resolve_bams(CHM13_BAM_MAP)

    # ── Resolve reference paths ─────────────────────────────────────
    hg002_ref = HG002_REF_PATH if os.path.exists(HG002_REF_PATH) else HG002_REF_PATH + '.gz'
    chm13_ref = CHM13_REF_PATH if os.path.exists(CHM13_REF_PATH) else CHM13_REF_PATH + '.gz'

    # ── Build unified job list ──────────────────────────────────────
    # Include ALL declared chemistry keys (even if BAM is missing) so that
    # existing checkpoints from deleted BAMs are still picked up.
    jobs = []  # (ref_label, chem_key, bam_path_or_None, ref_path_or_None)

    hg002_ref_ok = os.path.exists(hg002_ref)
    for chem_key in _all_declared_keys(HG002_BAM_MAP):
        bam_path = hg002_resolved.get(chem_key)  # None if BAM missing
        jobs.append(('hg002', chem_key, bam_path, hg002_ref if hg002_ref_ok else None))

    chm13_ref_ok = os.path.exists(chm13_ref)
    for chem_key in _all_declared_keys(CHM13_BAM_MAP):
        bam_path = chm13_resolved.get(chem_key)  # None if BAM missing
        jobs.append(('chm13', chem_key, bam_path, chm13_ref if chm13_ref_ok else None))

    if not jobs:
        raise RuntimeError("No chemistries declared — check BAM maps above.")

    print(f"\n{'═'*65}")
    print(f"  {len(jobs)} chemistry jobs declared")
    print(f"  FORCE_REGENERATE = {FORCE_REGENERATE}")
    print(f"{'═'*65}\n")

    common_kwargs = {
        'subsample': ERRORSMITH_SUBSAMPLE,
        'chroms': ERRORSMITH_CHROMS,
        'threads': ERRORSMITH_THREADS,
        'seed': SEED,
    }

    # ── Process each chemistry one at a time ────────────────────────
    t0 = time.time()
    completed = []   # newly generated this run
    skipped = []     # valid checkpoint already exists
    failed = []      # generation attempted but crashed
    no_bam = []      # BAM missing AND no checkpoint

    for job_idx, (ref_label, chem_key, bam_path, ref_path) in enumerate(jobs, 1):
        ckpt_fname = _checkpoint_name(ref_label, chem_key)
        ckpt_gdrive = os.path.join(GDRIVE_CHECKPOINT_DIR, ckpt_fname)

        # ── Check for existing checkpoint ────────────────────────
        if not FORCE_REGENERATE and _is_checkpoint_valid(ckpt_gdrive):
            row_count = _count_csv_rows(ckpt_gdrive)
            print(f"[{job_idx}/{len(jobs)}] ✓ SKIP {ref_label}/{chem_key} "
                  f"— checkpoint exists ({row_count:,} rows): {ckpt_fname}")
            skipped.append((ref_label, chem_key, ckpt_gdrive))
            continue

        # ── BAM or reference missing → can't generate ────────────
        if bam_path is None or ref_path is None:
            reason = "BAM not found" if bam_path is None else "reference not found"
            print(f"[{job_idx}/{len(jobs)}] ⏭ SKIP {ref_label}/{chem_key} "
                  f"— {reason}, no checkpoint")
            no_bam.append((ref_label, chem_key, reason))
            continue

        # ── Generate CSV for this single chemistry ───────────────
        print(f"\n[{job_idx}/{len(jobs)}] ▶ Processing {ref_label}/{chem_key}")
        print(f"  BAM: {os.path.basename(bam_path)}")
        print(f"  Ref: {ref_label} ({os.path.basename(ref_path)})")

        per_chem_out = Path(ERRORSMITH_OUT) / f'{ref_label}_{chem_key}'
        t_job = time.time()

        try:
            csv_path = generate_errorsmith_training_data(
                output_dir=per_chem_out,
                reference_path=ref_path,
                bam_map={chem_key: bam_path},
                **common_kwargs,
            )

            # Verify output
            if not os.path.exists(csv_path):
                raise FileNotFoundError(f"Expected CSV not found: {csv_path}")

            row_count = _count_csv_rows(csv_path)
            elapsed_job = time.time() - t_job

            # ── Checkpoint to Google Drive ───────────────────────
            shutil.copy2(csv_path, ckpt_gdrive)

            print(f"  ✓ {ref_label}/{chem_key}: {row_count:,} rows in {elapsed_job/60:.1f} min")
            print(f"  ✓ Checkpointed → {ckpt_fname}")
            completed.append((ref_label, chem_key, ckpt_gdrive))

        except Exception as e:
            elapsed_job = time.time() - t_job
            print(f"  ✗ FAILED {ref_label}/{chem_key} after {elapsed_job/60:.1f} min: {e}")
            traceback.print_exc()
            failed.append((ref_label, chem_key, str(e)))

        # ── Progress summary ─────────────────────────────────────
        done = len(completed) + len(skipped)
        remaining = len(jobs) - done - len(failed) - len(no_bam)
        elapsed_total = time.time() - t0
        print(f"  [{done} done / {remaining} remaining / {len(failed)} failed / "
              f"{len(no_bam)} no-BAM] Total elapsed: {elapsed_total/60:.1f} min")

    # ── Discover orphan checkpoint CSVs (BAM deleted / removed from map) ──
    # Scan the GDrive checkpoint dir for valid CSVs that weren't picked up
    # by the declared job list.  Chemistry is inferred from the filename:
    #   {ref_label}_{chem_key}.csv  →  e.g. hg002_ont_ulk114_r1041.csv
    _known_fnames = set()
    for ref_label, chem_key, _ in completed + skipped:
        _known_fnames.add(_checkpoint_name(ref_label, chem_key))
    for ref_label, chem_key, _ in no_bam:
        _known_fnames.add(_checkpoint_name(ref_label, chem_key))
    for ref_label, chem_key, _ in failed:
        _known_fnames.add(_checkpoint_name(ref_label, chem_key))

    orphans = []
    import re as _re_orphan
    for _f in sorted(os.listdir(GDRIVE_CHECKPOINT_DIR)):
        if not _f.endswith('.csv') or _f in _known_fnames:
            continue
        _orp_path = os.path.join(GDRIVE_CHECKPOINT_DIR, _f)
        if not _is_checkpoint_valid(_orp_path):
            continue
        # Parse chemistry from filename: {ref_label}_{chem_key}.csv
        _m = _re_orphan.match(r'^(hg002|chm13)_(.+)\.csv$', _f)
        if _m:
            _orp_ref, _orp_chem = _m.group(1), _m.group(2)
        else:
            # Fallback: use full stem, ref=unknown
            _orp_ref, _orp_chem = 'unknown', _f.replace('.csv', '')
        _orp_rows = _count_csv_rows(_orp_path)
        print(f"  🔍 Orphan checkpoint found: {_f} ({_orp_rows:,} rows)")
        print(f"     Chemistry (from filename): {_orp_ref}/{_orp_chem}")
        orphans.append((_orp_ref, _orp_chem, _orp_path))

    if orphans:
        print(f"\n  ℹ {len(orphans)} orphan checkpoint(s) recovered from GDrive")
        print(f"    (BAM deleted or removed from config — chemistry inferred from CSV name)")
    else:
        # No orphans found — all checkpoint CSVs accounted for
        pass

    # ── Streaming merge: concatenate CSVs without loading into RAM ───
    print(f"\n{'═'*65}")
    print(f"  Merging checkpoint CSVs (streaming — low memory)...")
    print(f"{'═'*65}")

    all_csvs = completed + skipped + orphans  # include orphan checkpoints too
    if not all_csvs:
        raise RuntimeError(
            f"No CSVs available to merge. "
            f"{len(failed)} job(s) failed, {len(no_bam)} missing BAMs "
            f"— check errors above."
        )

    total_rows = 0
    header_written = False

    with open(final_csv, 'w') as out_f:
        for ref_label, chem_key, csv_path in all_csvs:
            chunk_rows = 0
            with open(csv_path, 'r') as in_f:
                for line_idx, line in enumerate(in_f):
                    if line_idx == 0:
                        # Write header only once
                        if not header_written:
                            out_f.write(line)
                            header_written = True
                        continue
                    out_f.write(line)
                    chunk_rows += 1
            total_rows += chunk_rows
            print(f"  {ref_label}/{chem_key:30s}: {chunk_rows:>10,} rows")

    # ── Copy merged CSV to GDrive ────────────────────────────────────
    print(f"\n  Copying merged CSV to GDrive...")
    shutil.copy2(final_csv, GDRIVE_MERGED_CSV)
    sz_gb = os.path.getsize(GDRIVE_MERGED_CSV) / (1024**3)
    print(f"  ✓ Saved → {GDRIVE_MERGED_CSV} ({sz_gb:.2f} GB)")

    elapsed = time.time() - t0
    print(f"\n{'═'*65}")
    print(f"  ✅ Merged {len(all_csvs)} CSVs → {total_rows:,} total rows")
    print(f"  Output: {final_csv}")
    print(f"  Total elapsed: {elapsed/60:.1f} min")

    if failed:
        print(f"\n  ⚠ {len(failed)} chemistry job(s) FAILED:")
        for ref_label, chem_key, err in failed:
            print(f"    ✗ {ref_label}/{chem_key}: {err}")
        print(f"  Re-run this cell to retry failed jobs.")

    if no_bam:
        print(f"\n  ℹ {len(no_bam)} chemistry job(s) skipped (BAM/ref missing, no checkpoint):")
        for ref_label, chem_key, reason in no_bam:
            print(f"    ⏭ {ref_label}/{chem_key}: {reason}")

    if skipped:
        print(f"\n  ℹ {len(skipped)} chemistry job(s) loaded from checkpoint (cached).")
        print(f"    Set FORCE_REGENERATE = True to re-process all.")

    if orphans:
        print(f"\n  🔍 {len(orphans)} orphan checkpoint(s) recovered "
              f"(BAM deleted — chemistry inferred from CSV filename):")
        for ref_label, chem_key, _orp_path in orphans:
            print(f"    📎 {ref_label}/{chem_key}: {os.path.basename(_orp_path)}")
        print(f"    These are included in the merge.  To regenerate, "
              f"delete the CSV(s) and re-add the BAM(s).")

    print(f"{'═'*65}")

    # Point downstream cells at the final CSV
    errorsmith_csv = final_csv

Checkpoint dir: /content/drive/MyDrive/Colab_Training/errorsmith_checkpoints
✓ Merged CSV already exists on GDrive (150,035,641 rows, 31.73 GB)
  /content/drive/MyDrive/Colab_Training/errorsmith_checkpoints/errorsmith_training_merged.csv
  Copying to local workspace...
  ✓ Copied → /content/errorsmith_training/errorsmith_training.csv

  Set FORCE_REGENERATE = True to rebuild from scratch.


In [8]:
# ── Inspect ErrorSmith data (streaming — no full CSV load) ───────────
# Computes class distribution, chemistry breakdown, and feature stats
# in a single chunked pass so it works within Colab RAM limits.
from collections import Counter
from generate_errorsmith_training_data import CHEMISTRY_NAMES

ERROR_TYPE_NAMES = {0: 'correct', 1: 'substitution', 2: 'insertion', 3: 'deletion', 4: 'homopolymer_error'}
INSPECT_FEATURES = ['base_quality', 'mean_quality_window_5', 'position_in_read',
                    'gc_content_local', 'homopolymer_length', 'read_length']
CHUNK_SIZE = 500_000  # rows per chunk — ~200 MB peak RAM

t_inspect = time.time()
total_rows = 0
class_counts = Counter()
tech_counts = Counter()
tech_label_counts = Counter()
has_technology_col = None

# Online stats accumulators for feature summary (Welford's algorithm)
feat_n = {f: 0 for f in INSPECT_FEATURES}
feat_sum = {f: 0.0 for f in INSPECT_FEATURES}
feat_sum2 = {f: 0.0 for f in INSPECT_FEATURES}
feat_min = {f: float('inf') for f in INSPECT_FEATURES}
feat_max = {f: float('-inf') for f in INSPECT_FEATURES}

print(f"Scanning {errorsmith_csv} in {CHUNK_SIZE:,}-row chunks...")

for chunk in pd.read_csv(errorsmith_csv, chunksize=CHUNK_SIZE):
    total_rows += len(chunk)

    # Class distribution
    class_counts.update(chunk['error_type'].value_counts().to_dict())

    # Chemistry breakdown
    tech_counts.update(chunk['technology_encoded'].value_counts().to_dict())

    # Technology label (string column, optional)
    if has_technology_col is None:
        has_technology_col = 'technology' in chunk.columns
    if has_technology_col:
        tech_label_counts.update(chunk['technology'].value_counts().to_dict())

    # Feature running stats
    for feat in INSPECT_FEATURES:
        if feat in chunk.columns:
            col = chunk[feat].dropna()
            if len(col) > 0:
                feat_n[feat] += len(col)
                feat_sum[feat] += col.sum()
                feat_sum2[feat] += (col ** 2).sum()
                feat_min[feat] = min(feat_min[feat], col.min())
                feat_max[feat] = max(feat_max[feat], col.max())

    if total_rows % (CHUNK_SIZE * 5) == 0:
        print(f"  ... {total_rows:,} rows scanned")

elapsed_inspect = time.time() - t_inspect
print(f"\nShape: ({total_rows:,}, scanned via chunks)")
print(f"Scan time: {elapsed_inspect:.1f}s")

# Class distribution
print(f"\nClass distribution:")
for label in sorted(class_counts.keys()):
    count = class_counts[label]
    pct = count / total_rows * 100
    name = ERROR_TYPE_NAMES.get(int(label), f'unknown_{label}')
    print(f"  {int(label)} ({name:20s}): {count:>10,} ({pct:5.1f}%)")

# Chemistry breakdown
print(f"\nChemistry breakdown (technology_encoded):")
for code in sorted(tech_counts.keys()):
    count = tech_counts[code]
    chem_name = CHEMISTRY_NAMES.get(int(code), f'unknown_{code}')
    pct = count / total_rows * 100
    print(f"  {int(code)} ({chem_name:25s}): {count:>10,} ({pct:5.1f}%)")

if has_technology_col and tech_label_counts:
    print(f"\nTechnology label breakdown:")
    for label, count in sorted(tech_label_counts.items(), key=lambda x: -x[1]):
        print(f"  {label:35s}: {count:>10,}")

# Feature summary (mean, std, min, max from running stats)
print(f"\nFeature summary:")
print(f"  {'feature':30s} {'count':>10s} {'mean':>10s} {'std':>10s} {'min':>10s} {'max':>10s}")
print(f"  {'─'*30} {'─'*10} {'─'*10} {'─'*10} {'─'*10} {'─'*10}")
for feat in INSPECT_FEATURES:
    n = feat_n[feat]
    if n > 0:
        mean = feat_sum[feat] / n
        var = max(0, feat_sum2[feat] / n - mean ** 2)
        std = var ** 0.5
        print(f"  {feat:30s} {n:>10,} {mean:>10.3f} {std:>10.3f} "
              f"{feat_min[feat]:>10.3f} {feat_max[feat]:>10.3f}")
    else:
        print(f"  {feat:30s} {'—':>10s} {'—':>10s} {'—':>10s} {'—':>10s} {'—':>10s}")

Scanning /content/errorsmith_training/errorsmith_training.csv in 500,000-row chunks...
  ... 2,500,000 rows scanned
  ... 5,000,000 rows scanned
  ... 7,500,000 rows scanned
  ... 10,000,000 rows scanned
  ... 12,500,000 rows scanned
  ... 15,000,000 rows scanned
  ... 17,500,000 rows scanned
  ... 20,000,000 rows scanned
  ... 22,500,000 rows scanned
  ... 25,000,000 rows scanned
  ... 27,500,000 rows scanned
  ... 30,000,000 rows scanned
  ... 32,500,000 rows scanned
  ... 35,000,000 rows scanned
  ... 37,500,000 rows scanned
  ... 40,000,000 rows scanned
  ... 42,500,000 rows scanned
  ... 45,000,000 rows scanned
  ... 47,500,000 rows scanned
  ... 50,000,000 rows scanned
  ... 52,500,000 rows scanned
  ... 55,000,000 rows scanned
  ... 57,500,000 rows scanned
  ... 60,000,000 rows scanned
  ... 62,500,000 rows scanned
  ... 65,000,000 rows scanned
  ... 67,500,000 rows scanned
  ... 70,000,000 rows scanned
  ... 72,500,000 rows scanned
  ... 75,000,000 rows scanned
  ... 77,500,000

## 5. Train ErrorSmith Per-Chemistry Ensemble

**Per-chemistry family ensemble** — 4 XGBoost models, each specialised for a different error-rate regime.

| Family | Codes | Chemistries | ~Error Rate |
|--------|-------|-------------|-------------|
| `ont_high_error` | 1, 2 | ONT R9.4.1 (regular + ultralong) | 10–15% |
| `ont_modern` | 3, 4, 10, 11, 12 | ONT R10.4.1 (simplex, duplex, hiacc, dorado) | 3–8% |
| `lr_accurate` | 0, 9 | PacBio HiFi (Sequel II, Revio) | 0.1–0.5% |
| `short_read` | 5, 6, 7, 8, 13 | Illumina, Onso, Element (Aviti, UltraQ, Long) | 0.1–1% |

**Resumable 3-phase architecture** — each phase checkpoints per-family to GDrive:

| Phase | What it does | Checkpoint files |
|-------|-------------|-----------------|
| **Phase 1** | Per-family metadata scan + stratified sample + Optuna HP search | `errorsmith_phase1_{family}.pkl` |
| **Phase 2** | Per-family incremental full-data training | `errorsmith_phase2_{family}_booster.ubj` + `…_meta.json` |
| **Phase 3** | Per-family evaluation on true hold-out + model save | `error_classifier_{family}.ubj` / `.pkl` + `family_routing.json` |

> **After a disconnect**: re-run cells **5a → 5b → 5c** in order. Completed families/rounds are skipped.
> Set `FORCE_RETRAIN = True` at the top of 5a to discard all checkpoints and start fresh.

In [11]:
# ── 5a. Phase 1: Per-family metadata scan + Optuna HP search ─────────
#
# ARCHITECTURE: Per-chemistry family ensemble.
# 7 families × 1 XGBoost each = 7 specialised 5-class models.
#
# Classes: correct (0) | substitution (1) | insertion (2) |
#          deletion (3) | HP_error (4)
#
# Key design decisions:
#   - No chemistry indicator features (routing handles this)
#   - No interaction features (within-family model doesn't need them)
#   - technology_encoded retained for intra-family discrimination
#   - chemistry_error_rate / chemistry_hp_error_rate as informative priors
#   - True hold-out: 15% of each family's data reserved before Phase 2

import gc
import re as _re
import optuna
import pickle
import json
import time as _time
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)
from sklearn.utils import resample
from collections import Counter, defaultdict
from generate_errorsmith_training_data import CHEMISTRY_NAMES, CHEMISTRY_CODES

optuna.logging.set_verbosity(optuna.logging.WARNING)


# ═══════════════════════════════════════════════════════════════════════
#  CHEMISTRY FAMILY DEFINITIONS
# ═══════════════════════════════════════════════════════════════════════
FAMILIES = {
    'ont_r9': {
        'codes': [1, 2],
        'desc': 'ONT R9.4.1 (regular + ultralong)',
        'error_tier': 'high',
    },
    'ont_lsk114_r1041': {
        'codes': [3],
        'desc': 'ONT R10.4.1 ligation kit (simplex)',
        'error_tier': 'medium',
    },
    'ont_ulk114_r1041': {
        'codes': [4, 12],
        'desc': 'ONT R10.4.1 ultralong kit (simplex + Dorado)',
        'error_tier': 'medium-high',
    },
    'ont_hiacc': {
        'codes': [10, 11],
        'desc': 'ONT high-accuracy (duplex + hiacc)',
        'error_tier': 'low',
    },
    'lr_accurate': {
        'codes': [0, 9],
        'desc': 'PacBio HiFi (Sequel II, Revio)',
        'error_tier': 'low',
    },
    'short_read_mixed': {
        'codes': [5],
        'desc': 'Illumina (cycle-dependent, mixed error types)',
        'error_tier': 'low',
    },
    'short_read_sbb': {
        'codes': [6, 7, 8, 13],
        'desc': 'SBB/Element (Onso, Aviti, UltraQ, Long)',
        'error_tier': 'low',
    },
}

# Reverse lookup: chemistry code → family name
CHEMISTRY_TO_FAMILY = {}
for fam_name, fam_info in FAMILIES.items():
    for code in fam_info['codes']:
        CHEMISTRY_TO_FAMILY[code] = fam_name

# Known per-chemistry error rate priors (approximate, from literature/benchmarks)
# These give the model an informative prior about expected error rate
CHEMISTRY_ERROR_RATES = {
    0:  0.001,   # pacbio_hifi_sequel2 — ~Q30
    1:  0.130,   # ont_lsk110_r941 — ~13%
    2:  0.150,   # ont_ulk001_r941 — ~15% (ultralong, slightly worse)
    3:  0.050,   # ont_lsk114_r1041 — ~5%
    4:  0.060,   # ont_ulk114_r1041 — ~6%
    5:  0.003,   # illumina_hiseq2500 — ~Q25
    6:  0.002,   # pacbio_onso — ~Q27
    7:  0.003,   # element_aviti — ~Q25
    8:  0.001,   # element_ultraq — ~Q30
    9:  0.001,   # pacbio_hifi_revio — ~Q30
    10: 0.030,   # ont_r1041_duplex — ~3%
    11: 0.040,   # ont_ulk114_r1041_hiacc — ~4%
    12: 0.050,   # ont_ulk114_r1041_dorado — ~5%
    13: 0.002,   # element_aviti_lng — ~Q27
}

CHEMISTRY_HP_ERROR_RATES = {
    0:  0.005,   # pacbio_hifi — HP is main error mode
    1:  0.070,   # ont R9 — severe HP errors
    2:  0.080,   # ont R9 ultralong — worse in HP
    3:  0.025,   # ont R10 — much improved HP
    4:  0.030,   # ont R10 ultralong
    5:  0.001,   # illumina — minimal HP issues
    6:  0.0005,  # onso — SBB, very low HP
    7:  0.001,   # element — similar to illumina
    8:  0.0005,  # element ultraq — very low
    9:  0.003,   # revio — slightly better than sequel2
    10: 0.015,   # duplex — improved by consensus
    11: 0.020,   # hiacc — experimental
    12: 0.025,   # dorado — similar to simplex R10
    13: 0.001,   # element long — similar to standard
}


# ── Global trinucleotide substitution rate prior ────────────────────
# Populated during Pass 1 scan.  Maps trinuc_code (0-63) to the
# substitution rate observed in the full training dataset.  Gives the
# model a direct signal: "bases in this context are N% substitutions".
_TRINUC_SUB_RATES = {}           # trinuc_code -> sub rate (set in Pass 1)
_TRINUC_SUB_RATE_DEFAULT = 0.01  # fallback for missing codes / old runs

# ── Per-family feature set (no chemistry binary flags, no interactions) ──
FAMILY_FEATURES = [
    # Per-base features (16)
    'base_quality', 'mean_quality_window_5', 'mean_quality_window_20',
    'position_in_read', 'read_length',
    'gc_content_local', 'gc_content_read',
    'homopolymer_length', 'homopolymer_base', 'distance_to_hp',
    'trinucleotide_context', 'pentanucleotide_context',
    # Alignment quality
    'mapping_quality',
    # Reference context
    'ref_gc_window_50', 'ref_repeat_flag', 'ref_homopolymer_length',
    # Intra-family chemistry discrimination
    'technology_encoded',
    # Informative priors
    'chemistry_error_rate',
    'chemistry_hp_error_rate',
    # ── Multi-read k-mer spectrum features (from two-pass pipeline) ──
    'kmer_min_freq', 'kmer_mean_freq', 'kmer_solid_ratio', 'kmer_log_ratio',
    # ── Markov-chain context feature ──
    'markov_log_prob',
    # ── Cycle / position features (especially useful for short reads) ──
    'relative_position', 'cycle_error_prior', 'is_read2',
    # ── Substitution-detection derived features ──
    'bq_delta_5',            # base_quality - mean_quality_window_5
    'bq_delta_20',           # base_quality - mean_quality_window_20
    'ctx_center_base',       # center base identity from trinuc (0-3)
    'ctx_left_base',         # left flanking base from trinuc (0-3)
    'ctx_right_base',        # right flanking base from trinuc (0-3)
    'is_cpg_context',        # CpG deamination hotspot flag
    # ── Trinucleotide substitution prior ──
    'trinuc_sub_rate',       # population sub rate for this trinuc context
]

ES_LABEL = 'error_type'
ES_N_OPTUNA_TRIALS = 40           # per-family Optuna budget
ES_N_CV_FOLDS = 5

# ── Tunable constants ───────────────────────────────────────────────
HP_SAMPLE_PER_CLASS = 200_000     # per class per family (upper bound)
HP_RESAMPLE_MAX     = 200_000     # max balanced samples for HP search
MIN_PER_CHEM_CLASS  = 5000         # guaranteed minimum rows per (chem, class)
                                   # ensures ALL chemistries survive to evaluation

INCR_CHUNK_SIZE      = 2_000_000
INCR_RESAMPLE_MAX    = 400_000

EVAL_HOLDOUT_FRAC = 0.15

# ── GDrive checkpoint paths (per-family) ────────────────────────────
def _family_ckpt_paths(family_name):
    """Return dict of checkpoint paths for a family."""
    return {
        'phase1': os.path.join(GDRIVE_CHECKPOINT_DIR, f'errorsmith_phase1_{family_name}.pkl'),
        'phase2_booster': os.path.join(GDRIVE_CHECKPOINT_DIR, f'errorsmith_phase2_{family_name}_booster.ubj'),
        'phase2_meta': os.path.join(GDRIVE_CHECKPOINT_DIR, f'errorsmith_phase2_{family_name}_meta.json'),
    }

FORCE_RETRAIN = True   # Must be True when changing family definitions

# Per-family retrain override.  Set to a list of family names to retrain
# only those families (others load from checkpoint).  Set to None to use
# the global FORCE_RETRAIN flag for all families.
#   e.g.  RETRAIN_FAMILIES = ['ont_lsk114_r1041']   # only retrain ONT R10 simplex
#   e.g.  RETRAIN_FAMILIES = None             # honour FORCE_RETRAIN for all
RETRAIN_FAMILIES = None


def _remap_tech_code(code):
    """Remap out-of-range tech codes to the nearest valid chemistry.

    The training data generator assigns code = len(CHEMISTRY_CODES) for
    unknown technologies.  Map those back to a safe default (0 = PacBio
    HiFi Sequel II) so downstream lookups never KeyError.
    """
    if code in CHEMISTRY_ERROR_RATES:
        return code
    return 0  # fallback: PacBio HiFi Sequel II


# Multi-read features that come from the two-pass training pipeline CSV
_MULTI_READ_COLS = [
    'kmer_min_freq', 'kmer_mean_freq', 'kmer_solid_ratio', 'kmer_log_ratio',
    'markov_log_prob',
    'relative_position', 'cycle_error_prior', 'is_read2',
]


def _derive_features(chunk):
    """Add derived feature columns to a chunk."""
    # Remap ghost tech codes
    chunk['technology_encoded'] = chunk['technology_encoded'].map(
        lambda c: _remap_tech_code(c)).astype(int)

    # Ensure mapping_quality exists (backward compat with old CSVs)
    if 'mapping_quality' not in chunk.columns:
        chunk['mapping_quality'] = 30

    # Backward compat: zero-fill multi-read features absent from old CSVs
    for col in _MULTI_READ_COLS:
        if col not in chunk.columns:
            chunk[col] = 0.0

    # Add chemistry error rate priors
    chunk['chemistry_error_rate'] = chunk['technology_encoded'].map(
        CHEMISTRY_ERROR_RATES).fillna(0.01).astype(np.float32)
    chunk['chemistry_hp_error_rate'] = chunk['technology_encoded'].map(
        CHEMISTRY_HP_ERROR_RATES).fillna(0.01).astype(np.float32)


    # ── Substitution-detection derived features ──────────────────
    chunk['bq_delta_5']  = (chunk['base_quality'] - chunk['mean_quality_window_5']).astype(np.float32)
    chunk['bq_delta_20'] = (chunk['base_quality'] - chunk['mean_quality_window_20']).astype(np.float32)

    # Decompose trinucleotide ordinal (0-63) into base identities (0-3)
    _trinuc = chunk['trinucleotide_context'].fillna(-1).astype(int)
    _valid_trinuc = _trinuc.between(0, 63)
    chunk['ctx_center_base'] = np.where(_valid_trinuc, (_trinuc // 4) % 4, -1).astype(np.int8)
    chunk['ctx_left_base']   = np.where(_valid_trinuc, _trinuc // 16, -1).astype(np.int8)
    chunk['ctx_right_base']  = np.where(_valid_trinuc, _trinuc % 4, -1).astype(np.int8)

    # CpG deamination hotspot: center=C & right=G, or left=C & center=G
    chunk['is_cpg_context'] = (
        ((chunk['ctx_center_base'] == 1) & (chunk['ctx_right_base'] == 2)) |
        ((chunk['ctx_left_base'] == 1) & (chunk['ctx_center_base'] == 2))
    ).astype(np.int8)

    # Trinucleotide substitution rate prior (populated in Pass 1 scan)
    chunk['trinuc_sub_rate'] = chunk['trinucleotide_context'].fillna(-1).astype(int).map(
        _TRINUC_SUB_RATES).fillna(_TRINUC_SUB_RATE_DEFAULT).astype(np.float32)

    return chunk


def hybrid_resample(X, y, max_majority=100_000, random_state=42):
    """Balanced resampling: every class -> same target count.
    Fixes the old formula that kept the majority class huge while
    only raising minorities to the median, producing 35:1 imbalance
    for low-error-rate families (short_read) in the training CSV.
    """
    classes, counts = np.unique(y, return_counts=True)
    capped = [min(n, max_majority) for n in counts]
    # Single target for ALL classes: median of capped counts (floor 500)
    target = max(500, int(np.median(capped)))

    X_resampled, y_resampled = [], []
    for cls in classes:
        mask = y == cls
        X_cls, y_cls = X[mask], y[mask]
        if len(X_cls) > target:
            X_rs, y_rs = resample(X_cls, y_cls, n_samples=target,
                                  random_state=random_state, replace=False)
        elif len(X_cls) < target:
            X_rs, y_rs = resample(X_cls, y_cls, n_samples=target,
                                  random_state=random_state, replace=True)
        else:
            X_rs, y_rs = X_cls, y_cls
        X_resampled.append(X_rs)
        y_resampled.append(y_rs)
    return np.vstack(X_resampled), np.concatenate(y_resampled)


def compute_class_weights(y):
    """Inverse-frequency class weights for DMatrix."""
    _, inverse, counts = np.unique(y, return_inverse=True, return_counts=True)
    weights = (1.0 / counts[inverse]).astype(np.float32)
    weights *= len(y) / weights.sum()
    return weights


# ── Optuna objective: 5-class model for one family ──────────────────
def es_optuna_objective(trial, X, y):
    """5-class XGBoost classifier HP search."""
    params = {
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 200, 1500),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 15),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'gamma': trial.suggest_float('gamma', 0, 5.0),
        'tree_method': XGB_TREE,
        'device': XGB_DEVICE,
        'objective': 'multi:softprob',
        'num_class': 5,
        'eval_metric': 'mlogloss',
        'random_state': SEED,
    }
    model = xgb.XGBClassifier(**params)
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    scores = cross_val_score(model, X, y, cv=skf, scoring='f1_macro')
    return scores.mean()

# ── Columns to read from CSV ────────────────────────────────────────
_probe = pd.read_csv(errorsmith_csv, nrows=2)
_csv_cols = set(_probe.columns)
del _probe; gc.collect()

# We need the base features + label + tech code from CSV
# (chemistry_error_rate / chemistry_hp_error_rate are derived on-the-fly)
_want = set(FAMILY_FEATURES) | {ES_LABEL, 'technology_encoded'}
# Derived features (computed in _derive_features, not read from CSV)
_want -= {'chemistry_error_rate', 'chemistry_hp_error_rate',
          'bq_delta_5', 'bq_delta_20', 'ctx_center_base',
          'ctx_left_base', 'ctx_right_base', 'is_cpg_context',
          'trinuc_sub_rate'}
_usecols = sorted(c for c in _want if c in _csv_cols)


# ═══════════════════════════════════════════════════════════════════════
#         PASS 1: Metadata scan (chemistry × class counts)
# ═══════════════════════════════════════════════════════════════════════
print("=" * 60)
print("PASS 1: Scanning metadata (chemistry " + chr(215) + " class distribution)")
print("=" * 60)
_chem_class_counts = defaultdict(Counter)
_class_totals = Counter()
_unique_chems = set()

# Also accumulate per-trinucleotide substitution counts for trinuc_sub_rate
_trinuc_total_arr = np.zeros(64, dtype=np.int64)
_trinuc_sub_arr   = np.zeros(64, dtype=np.int64)

for _mchunk in pd.read_csv(errorsmith_csv,
                            usecols=['technology_encoded', ES_LABEL, 'trinucleotide_context'],
                            chunksize=1_000_000):
    for (tech, cls), cnt in _mchunk.groupby(['technology_encoded', ES_LABEL]).size().items():
        chem = _remap_tech_code(tech)
        _chem_class_counts[chem][cls] += cnt
        _class_totals[cls] += cnt
        _unique_chems.add(chem)
    # Accumulate per-trinucleotide substitution counts
    _tc_vals = _mchunk['trinucleotide_context'].fillna(-1).astype(int).values
    _valid_tc = (_tc_vals >= 0) & (_tc_vals <= 63)
    _trinuc_total_arr += np.bincount(_tc_vals[_valid_tc], minlength=64)
    _sub_mask = _valid_tc & (_mchunk[ES_LABEL].values == 1)
    if _sub_mask.any():
        _trinuc_sub_arr += np.bincount(_tc_vals[_sub_mask], minlength=64)

n_raw = sum(_class_totals.values())
del _mchunk; gc.collect()

# Compute global trinucleotide -> substitution rate prior
_overall_sub_rate = float(_trinuc_sub_arr.sum() / max(_trinuc_total_arr.sum(), 1))
_TRINUC_SUB_RATE_DEFAULT = _overall_sub_rate
_TRINUC_SUB_RATES = {}
for _tc_code in range(64):
    if _trinuc_total_arr[_tc_code] > 100:
        _TRINUC_SUB_RATES[_tc_code] = float(
            _trinuc_sub_arr[_tc_code] / _trinuc_total_arr[_tc_code])
    else:
        _TRINUC_SUB_RATES[_tc_code] = _overall_sub_rate
del _trinuc_total_arr, _trinuc_sub_arr
print(f"  Trinuc sub rates: min={min(_TRINUC_SUB_RATES.values()):.4f}, "
      f"max={max(_TRINUC_SUB_RATES.values()):.4f}, overall={_overall_sub_rate:.4f}")

print(f"  Total rows: {n_raw:,}")
print(f"  Classes: {dict(sorted(_class_totals.items()))}")
print(f"  Chemistries: {sorted(_unique_chems)}")

# Show per-family counts
for fam_name, fam_info in FAMILIES.items():
    fam_codes = set(fam_info['codes'])
    fam_total = sum(_chem_class_counts[c].get(cls, 0)
                    for c in fam_codes & _unique_chems
                    for cls in range(5))
    fam_chems_present = fam_codes & _unique_chems
    print(f"\n  {fam_name} ({fam_info['desc']}):")
    print(f"    Codes present: {sorted(fam_chems_present)} ({len(fam_chems_present)}/{len(fam_codes)})")
    print(f"    Total rows: {fam_total:,}")
    for cc in sorted(fam_chems_present):
        name = CHEMISTRY_NAMES.get(int(cc), f'unk_{cc}')
        counts = _chem_class_counts[cc]
        total = sum(counts.values())
        print(f"      {name:35s}: {total:>10,}  {dict(sorted(counts.items()))}")


# ═══════════════════════════════════════════════════════════════════════
#  PHASE 1: Per-family Optuna HP search
# ═══════════════════════════════════════════════════════════════════════
# Results stored in family_results dict
family_results = {}

# Determine which families have data
active_families = []
for fam_name, fam_info in FAMILIES.items():
    fam_codes = set(fam_info['codes']) & _unique_chems
    if fam_codes:
        active_families.append(fam_name)
    else:
        print(f"\n  " + chr(9888) + f" Skipping {fam_name}: no chemistries present in data")

print(f"\n{'=' * 60}")
print(f"PHASE 1: Per-family HP search ({len(active_families)} families)")
print(f"  Families: {active_families}")
print(f"  Optuna: {ES_N_OPTUNA_TRIALS} trials per family")
print(f"{'=' * 60}")


for fam_name in active_families:
    fam_info = FAMILIES[fam_name]
    fam_codes = set(fam_info['codes']) & _unique_chems
    ckpts = _family_ckpt_paths(fam_name)
    _fam_ready = False

    print(f"\n{'─' * 60}")
    print(f"  Family: {fam_name} ({fam_info['desc']})")
    print(f"  Codes:  {sorted(fam_codes)}")
    print(f"{'─' * 60}")

    # ── Check for existing Phase 1 checkpoint ───────────────────
    _force_this = (RETRAIN_FAMILIES is not None and fam_name in RETRAIN_FAMILIES) or \
                  (RETRAIN_FAMILIES is None and FORCE_RETRAIN)
    if not _force_this and os.path.exists(ckpts['phase1']):
        try:
            with open(ckpts['phase1'], 'rb') as f:
                _p1 = pickle.load(f)
            _p1_arch = _p1.get('architecture', '')
            if _p1_arch != 'per_family':
                print(f"    " + chr(9888) + f" Old checkpoint (arch={_p1_arch}) — forcing re-run")
            else:
                best_params = _p1['best_params']
                X_sample   = _p1['X_sample']
                y_sample   = _p1['y_sample']
                t_sample   = _p1['t_sample']
                n_fam_raw  = _p1['n_fam_raw']
                _p1_f1     = _p1.get('best_value', 'N/A')
                _fam_ready = True
                print(f"    " + chr(10003) + f" Loaded Phase 1 checkpoint")
                print(f"      Best F1-macro: {_p1_f1}")
                print(f"      Sample: {X_sample.shape[0]:,} " + chr(215) + f" {X_sample.shape[1]}")
            del _p1; gc.collect()
        except Exception as e:
            print(f"    " + chr(9888) + f" Checkpoint load failed: {e}")

    if not _fam_ready:
        # ── Collect stratified sample for this family ────────────
        # Chemistry × class quotas to ensure EVERY chemistry gets
        # represented — prevents rare platforms from being swamped.
        print(f"    Collecting stratified sample (chem×class quotas)...")

        _reservoirs = defaultdict(list)   # (chem, cls) → [arrays]
        _collected  = Counter()           # (chem, cls) → count
        _t_reservoirs = defaultdict(list) # (chem, cls) → [tech arrays]

        # Build per-(chem, cls) quota: proportional share with guaranteed floor
        _fam_n_chems = len(fam_codes)
        _per_chem_budget = max(MIN_PER_CHEM_CLASS,
                               HP_SAMPLE_PER_CLASS // max(_fam_n_chems, 1))
        _chem_class_quota = {}
        for cc in fam_codes:
            for cls in range(5):
                n_avail = _chem_class_counts.get(cc, {}).get(cls, 0)
                # At least MIN_PER_CHEM_CLASS, up to fair share, capped by available
                quota = min(max(MIN_PER_CHEM_CLASS, _per_chem_budget), n_avail)
                _chem_class_quota[(cc, cls)] = quota

        _total_quota = sum(_chem_class_quota.values())
        print(f"    Quotas: {_fam_n_chems} chems × 5 classes = "
              f"{_total_quota:,} target rows  "
              f"(floor={MIN_PER_CHEM_CLASS}/chem/class)")

        for chunk in pd.read_csv(errorsmith_csv, usecols=_usecols, chunksize=500_000):
            chunk = _derive_features(chunk)
            t_chunk = chunk['technology_encoded'].values.astype(int)

            # Filter to this family's codes
            fam_mask = np.isin(t_chunk, list(fam_codes))
            if fam_mask.sum() == 0:
                continue

            chunk_fam = chunk.loc[fam_mask]
            X_chunk = chunk_fam[FAMILY_FEATURES].fillna(0).values.astype(np.float32)
            y_chunk = chunk_fam[ES_LABEL].values.astype(np.int8)
            t_fam   = chunk_fam['technology_encoded'].values.astype(np.int8)

            # Collect per (chemistry, class)
            for cc in fam_codes:
                chem_mask = t_fam == cc
                if chem_mask.sum() == 0:
                    continue
                for cls in range(5):
                    key = (cc, cls)
                    quota = _chem_class_quota.get(key, 0)
                    if _collected[key] >= quota:
                        continue
                    remaining = quota - _collected[key]
                    cls_mask = chem_mask & (y_chunk == cls)
                    n_hit = int(cls_mask.sum())
                    if n_hit == 0:
                        continue
                    take = min(remaining, n_hit)
                    idx = np.where(cls_mask)[0][:take]
                    _reservoirs[key].append(X_chunk[idx])
                    _t_reservoirs[key].append(t_fam[idx])
                    _collected[key] += take

            # Early stop: all quotas met
            if all(_collected.get(k, 0) >= _chem_class_quota.get(k, 0)
                   for k in _chem_class_quota):
                print(f"    All chem×class quotas met " + chr(8212) + " stopping early")
                break

        del chunk, X_chunk, y_chunk; gc.collect()

        # Report per-chemistry collection
        for cc in sorted(fam_codes):
            cname = CHEMISTRY_NAMES.get(int(cc), f'unk_{cc}')
            cc_total = sum(_collected.get((cc, cls), 0) for cls in range(5))
            cc_quota = sum(_chem_class_quota.get((cc, cls), 0) for cls in range(5))
            fill_pct = cc_total / max(cc_quota, 1) * 100
            print(f"      {cname:35s}: {cc_total:>8,} / {cc_quota:>8,} ({fill_pct:.0f}%)")

        # Merge into flat arrays
        X_parts, y_parts, t_parts = [], [], []
        for key in sorted(_reservoirs.keys()):
            cc, cls = key
            xs = _reservoirs[key]
            ts = _t_reservoirs[key]
            if xs:
                xv = np.vstack(xs)
                tv = np.concatenate(ts)
                yv = np.full(len(xv), cls, dtype=np.int8)
                X_parts.append(xv)
                y_parts.append(yv)
                t_parts.append(tv)
        del _reservoirs, _t_reservoirs; gc.collect()

        if not X_parts:
            print(f"    " + chr(9888) + f" No data collected for {fam_name} — skipping")
            continue

        X_sample = np.vstack(X_parts).astype(np.float32)
        y_sample = np.concatenate(y_parts).astype(np.int8)
        t_sample = np.concatenate(t_parts).astype(np.int8)
        del X_parts, y_parts, t_parts; gc.collect()

        n_fam_raw = sum(
            sum(_chem_class_counts[cc].values()) for cc in fam_codes
        )

        # Verify ALL chemistries survived sampling
        _sampled_chems = set(int(c) for c in np.unique(t_sample))
        _missing_chems = fam_codes - _sampled_chems
        if _missing_chems:
            _missing_names = [CHEMISTRY_NAMES.get(c, f'unk_{c}') for c in sorted(_missing_chems)]
            print(f"    " + chr(9888) + f" WARNING: {len(_missing_chems)} chemistries have ZERO "
                  f"rows in sample: {_missing_names}")
        else:
            print(f"    " + chr(10003) + f" All {len(fam_codes)} chemistries represented in sample")

        print(f"    Sample: {X_sample.shape[0]:,} " + chr(215) + f" {X_sample.shape[1]} features")
        print(f"    Per-class: {dict(Counter(int(c) for c in y_sample))}")

        # Class-balanced resampling for HP search
        _max_per_class = max(1000, HP_RESAMPLE_MAX // max(len(np.unique(y_sample)), 1))
        X_hp, y_hp = hybrid_resample(
            X_sample, y_sample,
            max_majority=_max_per_class, random_state=SEED)

        print(f"    HP search: {X_hp.shape[0]:,} balanced samples")
        print(f"    Balanced: {dict(Counter(int(c) for c in y_hp))}")

        # ── Optuna ──────────────────────────────────────────────
        print(f"    " + chr(9484) + f" Optuna: {ES_N_OPTUNA_TRIALS} trials...")
        _t0 = _time.time()
        study = optuna.create_study(direction='maximize')
        study.optimize(
            lambda trial: es_optuna_objective(trial, X_hp, y_hp),
            n_trials=ES_N_OPTUNA_TRIALS, show_progress_bar=True)

        best_params = study.best_params.copy()
        best_params['tree_method'] = XGB_TREE
        best_params['device'] = XGB_DEVICE
        best_params['objective'] = 'multi:softprob'
        best_params['num_class'] = 5
        best_params['eval_metric'] = 'mlogloss'
        best_params['random_state'] = SEED
        _optuna_time = _time.time() - _t0

        _p1_f1 = study.best_value
        print(f"    " + chr(9474) + f"  Best F1-macro: {_p1_f1:.4f}")
        print(f"    " + chr(9474) + f"  depth={best_params['max_depth']}, "
              f"lr={best_params['learning_rate']:.4f}, "
              f"n_est={best_params['n_estimators']}")
        print(f"    " + chr(9492) + f" Done ({_optuna_time / 60:.1f} min)")

        del X_hp, y_hp; gc.collect()

        # ── Save Phase 1 checkpoint ─────────────────────────────
        _p1_bundle = {
            'architecture': 'per_family',
            'family': fam_name,
            'best_params': best_params,
            'best_value': float(_p1_f1),
            'X_sample': X_sample,
            'y_sample': y_sample,
            't_sample': t_sample,
            'n_fam_raw': n_fam_raw,
            'feature_names': FAMILY_FEATURES,
            '_chem_class_counts': {int(k): dict(v) for k, v in _chem_class_counts.items()},
            '_class_totals': dict(_class_totals),
            'trinuc_sub_rates': _TRINUC_SUB_RATES,
            'trinuc_sub_rate_default': _TRINUC_SUB_RATE_DEFAULT,
        }
        with open(ckpts['phase1'], 'wb') as f:
            pickle.dump(_p1_bundle, f, protocol=pickle.HIGHEST_PROTOCOL)
        del _p1_bundle
        _sz = os.path.getsize(ckpts['phase1']) / (1024**2)
        print(f"    " + chr(10003) + f" Checkpoint saved ({_sz:.1f} MB)")

    # ── Store results for this family ───────────────────────────
    family_results[fam_name] = {
        'best_params': best_params,
        'X_sample': X_sample,
        'y_sample': y_sample,
        't_sample': t_sample,
        'n_fam_raw': n_fam_raw,
        'p1_f1': _p1_f1 if isinstance(_p1_f1, float) else 0.0,
        'study': study if 'study' in locals() and study is not None else None,
    }

    del X_sample, y_sample, t_sample; gc.collect()

print(f"\n{'=' * 60}")
print(f"  Phase 1 complete for {len(family_results)} families:")
for fn, fr in family_results.items():
    print(f"    {fn:20s}  F1={fr['p1_f1']:.4f}  "
          f"depth={fr['best_params'].get('max_depth', '?')}")

print(f"  Proceed to Phase 2.")
print(f"{'=' * 60}")

PASS 1: Scanning metadata (chemistry × class distribution)
  Trinuc sub rates: min=0.0839, max=0.3408, overall=0.1237
  Total rows: 150,035,641
  Classes: {0: 102007163, 1: 19073258, 2: 6460332, 3: 8995164, 4: 13499724}
  Chemistries: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]

  ont_r9 (ONT R9.4.1 (regular + ultralong)):
    Codes present: [1, 2] (2/2)
    Total rows: 20,001,431
      ont_lsk110_r941                    : 10,001,065  {0: 1977996, 1: 1813960, 2: 1371610, 3: 1978199, 4: 2859300}
      ont_ulk001_r941                    : 10,000,366  {0: 2056509, 1: 1866784, 2: 1126333, 3: 2226072, 4: 2724668}

  ont_lsk114_r1041 (ONT R10.4.1 ligation kit (simplex)):
    Codes present: [3] (1/1)
    Total rows: 10,000,008
      ont_lsk114_r1041                   : 10,000,008  {0: 5049281, 1: 1674807, 2: 671332, 3: 1052699, 4: 1551889}

  ont_ulk114_r1041 (ONT R10.4.1 ultralong kit (simplex + Dorado)):
    Codes present: [4, 12] (2/2)
    Total rows: 20,031,383
      ont_ulk114_r1041  

  0%|          | 0/40 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [03:46:25] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


    │  Best F1-macro: 0.8806
    │  depth=10, lr=0.0617, n_est=627
    └ Done (10.9 min)
    ✓ Checkpoint saved (131.6 MB)

────────────────────────────────────────────────────────────
  Family: ont_lsk114_r1041 (ONT R10.4.1 ligation kit (simplex))
  Codes:  [3]
────────────────────────────────────────────────────────────
    Quotas: 1 chems × 5 classes = 1,000,000 target rows  (floor=5000/chem/class)
    All chem×class quotas met — stopping early
      ont_lsk114_r1041                   : 1,000,000 / 1,000,000 (100%)
    ✓ All 1 chemistries represented in sample
    Sample: 1,000,000 × 34 features
    Per-class: {0: 200000, 1: 200000, 2: 200000, 3: 200000, 4: 200000}
    HP search: 200,000 balanced samples
    Balanced: {0: 40000, 1: 40000, 2: 40000, 3: 40000, 4: 40000}
    ┌ Optuna: 40 trials...


  0%|          | 0/40 [00:00<?, ?it/s]

...

    │  Best F1-macro: 0.9347
    │  depth=10, lr=0.0251, n_est=1101
    └ Done (19.4 min)
    ✓ Checkpoint saved (121.2 MB)

────────────────────────────────────────────────────────────
  Family: short_read_mixed (Illumina (cycle-dependent, mixed error types))
  Codes:  [5]
────────────────────────────────────────────────────────────
    Quotas: 1 chems × 5 classes = 826,113 target rows  (floor=5000/chem/class)
    All chem×class quotas met — stopping early
      illumina_hiseq2500                 :  826,113 /  826,113 (100%)
    ✓ All 1 chemistries represented in sample
    Sample: 826,113 × 34 features
    Per-class: {0: 200000, 1: 200000, 2: 200000, 3: 26113, 4: 200000}
    HP search: 200,000 balanced samples
    Balanced: {0: 40000, 1: 40000, 2: 40000, 3: 40000, 4: 40000}
    ┌ Optuna: 40 trials...


  0%|          | 0/40 [00:00<?, ?it/s]

    │  Best F1-macro: 0.9807
    │  depth=8, lr=0.0233, n_est=1186
    └ Done (11.6 min)
    ✓ Checkpoint saved (108.7 MB)

────────────────────────────────────────────────────────────
  Family: short_read_sbb (SBB/Element (Onso, Aviti, UltraQ, Long))
  Codes:  [6, 7, 8, 13]
────────────────────────────────────────────────────────────
    Quotas: 4 chems × 5 classes = 491,165 target rows  (floor=5000/chem/class)
    All chem×class quotas met — stopping early
      pacbio_onso                        :  110,589 /  110,589 (100%)
      element_aviti                      :  134,973 /  134,973 (100%)
      element_ultraq                     :  125,382 /  125,382 (100%)
      element_aviti_lng                  :  120,221 /  120,221 (100%)
    ✓ All 4 chemistries represented in sample
    Sample: 491,165 × 34 features
    Per-class: {0: 200000, 1: 200000, 2: 3695, 3: 4440, 4: 83030}
    HP search: 200,000 balanced samples
    Balanced: {0: 40000, 1: 40000, 2: 40000, 3: 40000, 4: 40000}
    ┌ 

  0%|          | 0/40 [00:00<?, ?it/s]

    │  Best F1-macro: 0.9900
    │  depth=5, lr=0.2076, n_est=1415
    └ Done (7.3 min)
    ✓ Checkpoint saved (64.6 MB)

  Phase 1 complete for 7 families:
    ont_r9                F1=0.8806  depth=10
    ont_lsk114_r1041      F1=0.8460  depth=12
    ont_ulk114_r1041      F1=0.9040  depth=11
    ont_hiacc             F1=0.9036  depth=10
    lr_accurate           F1=0.9347  depth=10
    short_read_mixed      F1=0.9807  depth=8
    short_read_sbb        F1=0.9900  depth=5
  Proceed to Phase 2.


### 5b. Phase 2 — Per-Family Incremental Full-Data Training

Each family's XGBoost booster is trained incrementally on **only its own chemistries' data**.
Tree budget from Optuna is distributed across chunks. Checkpoints after every round.

In [12]:
# ── 5b. Phase 2: Per-family incremental full-data training ────────────

import gc, json, math, pickle, time as _time
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import f1_score

print(f"{'=' * 60}")
print(f"PHASE 2: Per-family incremental training")
print(f"{'=' * 60}")

for fam_name in family_results:
    fam_info = FAMILIES[fam_name]
    fam_codes = set(fam_info['codes']) & _unique_chems
    fr = family_results[fam_name]
    best_params = fr['best_params']
    n_fam_raw = fr['n_fam_raw']
    ckpts = _family_ckpt_paths(fam_name)

    _n_est = best_params.get('n_estimators', 600)
    _n_chunks = max(1, math.ceil(n_fam_raw / INCR_CHUNK_SIZE))
    _trees_per_chunk = max(1, math.ceil(_n_est / _n_chunks))

    print(f"\n{'─' * 60}")
    print(f"  Family: {fam_name}")
    print(f"  Data: {n_fam_raw:,} rows, {_n_chunks} chunks, "
          f"{_trees_per_chunk} trees/chunk (total {_n_est})")
    print(f"{'─' * 60}")

    # Build param dict for xgb.train()
    _incr_params = {k: v for k, v in best_params.items() if k != 'n_estimators'}
    if 'random_state' in _incr_params:
        _incr_params['seed'] = _incr_params.pop('random_state')

    _booster = None
    _total_trees = 0
    _total_rows_seen = 0
    _round_num = 0
    _round_f1s = []
    _chunks_to_skip = 0

    # ── Check Phase 2 checkpoint ────────────────────────────────
    _force_this = (RETRAIN_FAMILIES is not None and fam_name in RETRAIN_FAMILIES) or \
                  (RETRAIN_FAMILIES is None and FORCE_RETRAIN)
    if (not _force_this
        and os.path.exists(ckpts['phase2_booster'])
        and os.path.exists(ckpts['phase2_meta'])):
        try:
            with open(ckpts['phase2_meta'], 'r') as f:
                _p2_meta = json.load(f)
            _booster = xgb.Booster()
            _booster.load_model(ckpts['phase2_booster'])
            _round_num       = _p2_meta['round_num']
            _total_rows_seen = _p2_meta['total_rows_seen']
            _total_trees     = _p2_meta.get('total_trees', 0)
            _round_f1s       = _p2_meta.get('round_f1s', [])
            _chunks_to_skip  = _round_num

            if _total_rows_seen >= n_fam_raw:
                print(f"    " + chr(10003) + f" Already complete ({_round_num} rounds, "
                      f"{_total_rows_seen:,} rows, {_total_trees} trees)")
                fr['booster'] = _booster
                fr['total_trees'] = _total_trees
                fr['round_f1s'] = _round_f1s
                fr['round_num'] = _round_num
                fr['trees_per_chunk'] = _trees_per_chunk
                continue
            else:
                print(f"    " + chr(10003) + f" Resuming from round {_round_num} "
                      f"({_total_rows_seen:,}/{n_fam_raw:,})")
        except Exception as e:
            print(f"    " + chr(9888) + f" Checkpoint load failed: {e}")
            _booster = None; _chunks_to_skip = _round_num = 0
            _total_rows_seen = _total_trees = 0; _round_f1s = []

    # ── Stream family's data ────────────────────────────────────
    _t0 = _time.time()
    _fam_chunk_idx = 0

    for _chunk_idx, _raw_chunk in enumerate(
        pd.read_csv(errorsmith_csv, usecols=_usecols, chunksize=INCR_CHUNK_SIZE)
    ):
        _raw_chunk = _derive_features(_raw_chunk)
        t_all = _raw_chunk['technology_encoded'].values.astype(int)

        # Filter to this family
        fam_mask = np.isin(t_all, list(fam_codes))
        if fam_mask.sum() == 0:
            del _raw_chunk
            continue

        _fam_chunk_idx += 1
        if _fam_chunk_idx <= _chunks_to_skip:
            if _fam_chunk_idx == 1 or _fam_chunk_idx == _chunks_to_skip:
                print(f"    Round {_fam_chunk_idx}: " + chr(9193) + " skip")
            elif _fam_chunk_idx == 2:
                print(f"    ... skipping ...")
            del _raw_chunk
            continue

        _round_num += 1
        chunk_fam = _raw_chunk.loc[fam_mask]
        X_rnd = chunk_fam[FAMILY_FEATURES].fillna(0).values.astype(np.float32)
        y_rnd = chunk_fam[ES_LABEL].values.astype(np.int8)
        _total_rows_seen += len(y_rnd)
        del _raw_chunk, chunk_fam

        # Class-balanced resampling
        _max_cls = max(500, INCR_RESAMPLE_MAX // max(len(np.unique(y_rnd)), 1))
        X_bal, y_bal = hybrid_resample(X_rnd, y_rnd,
                                       max_majority=_max_cls,
                                       random_state=SEED + _round_num)

        # Diagnostic: verify balanced resampling
        _bal_cls, _bal_cnt = np.unique(y_bal, return_counts=True)
        _bal_str = ", ".join(f"c{c}={n:,}" for c, n in zip(_bal_cls, _bal_cnt))
        if _round_num <= 2 or _round_num == _n_chunks:
            print(f"      Resampled: {_bal_str}  (total {len(y_bal):,})")

        _rnd_weights = compute_class_weights(y_bal)
        del X_rnd, y_rnd; gc.collect()

        # Train
        dtrain = xgb.DMatrix(X_bal, label=y_bal, weight=_rnd_weights)
        _booster = xgb.train(
            params=_incr_params,
            dtrain=dtrain,
            num_boost_round=_trees_per_chunk,
            xgb_model=_booster,
            verbose_eval=False,
        )
        _total_trees = _booster.num_boosted_rounds()

        # Quick eval
        _eval_n = min(10_000, len(y_bal))
        _eval_rng = np.random.RandomState(SEED + _round_num)
        _eval_idx = _eval_rng.permutation(len(y_bal))[:_eval_n]
        _dm_eval = xgb.DMatrix(X_bal[_eval_idx])
        _probs = _booster.predict(_dm_eval)
        _preds = _probs.reshape(-1, 5).argmax(axis=1) if _probs.ndim == 1 else _probs.argmax(axis=1)
        _rnd_f1 = f1_score(y_bal[_eval_idx], _preds, average='macro', zero_division=0)
        _round_f1s.append(float(_rnd_f1))

        print(f"    Round {_round_num:>2}: {len(y_bal):>8,} rows, "
              f"{_total_trees} trees, F1={_rnd_f1:.4f}  "
              f"[{_total_rows_seen:,}/{n_fam_raw:,}]")

        del X_bal, y_bal, _rnd_weights, dtrain; gc.collect()

        # Checkpoint
        _booster.save_model(ckpts['phase2_booster'])
        with open(ckpts['phase2_meta'], 'w') as f:
            json.dump({
                'family': fam_name,
                'round_num': _round_num,
                'total_rows_seen': _total_rows_seen,
                'total_trees': _total_trees,
                'round_f1s': _round_f1s,
                'chunk_size': INCR_CHUNK_SIZE,
                'n_fam_raw': n_fam_raw,
                'trees_per_chunk': _trees_per_chunk,
                'complete': _total_rows_seen >= n_fam_raw,
            }, f, indent=2)

    _incr_time = _time.time() - _t0
    print(f"    " + chr(10003) + f" Done: {_total_trees} trees, "
          f"{_total_rows_seen:,} rows, {_incr_time / 60:.1f} min")

    fr['booster'] = _booster
    fr['total_trees'] = _total_trees
    fr['round_f1s'] = _round_f1s
    fr['round_num'] = _round_num
    fr['trees_per_chunk'] = _trees_per_chunk

print(f"\n{'=' * 60}")

print(f"  Phase 2 complete. Proceed to Phase 3.")
print(f"{'=' * 60}")

PHASE 2: Per-family incremental training

────────────────────────────────────────────────────────────
  Family: ont_r9
  Data: 20,001,431 rows, 11 chunks, 57 trees/chunk (total 627)
────────────────────────────────────────────────────────────
      Resampled: c0=80,000, c1=80,000, c2=80,000, c3=80,000, c4=80,000  (total 400,000)
    Round  1:  400,000 rows, 57 trees, F1=0.8624  [1,967,714/20,001,431]
      Resampled: c0=80,000, c1=80,000, c2=80,000, c3=80,000, c4=80,000  (total 400,000)
    Round  2:  400,000 rows, 114 trees, F1=0.8351  [3,967,714/20,001,431]
    Round  3:  400,000 rows, 171 trees, F1=0.8658  [5,967,714/20,001,431]
    Round  4:  400,000 rows, 228 trees, F1=0.8799  [7,967,714/20,001,431]
    Round  5:  400,000 rows, 285 trees, F1=0.8394  [9,967,714/20,001,431]
    Round  6:  400,000 rows, 342 trees, F1=0.8446  [11,967,714/20,001,431]
    Round  7:  400,000 rows, 399 trees, F1=0.8364  [13,967,714/20,001,431]
    Round  8:  400,000 rows, 456 trees, F1=0.8655  [15,967,71

### 5c. Phase 3 — Per-Family Evaluation + Model Save

**True hold-out evaluation:** 15% of each family's Phase 1 sample is reserved
for evaluation. The booster has **never seen** these rows during Phase 2 training
(Phase 2 trains on the full CSV, but this hold-out was drawn from the Phase 1
stratified sample and is guaranteed to be different from the Phase 2 balanced chunks).

Saves per-family models + routing JSON for inference-time dispatch.

In [13]:
# ── 5c. Phase 3: Per-family evaluation + model save ───────────────────

import gc, os, json, pickle, time as _time
import numpy as np
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.isotonic import IsotonicRegression

print(f"{'=' * 60}")
print(f"PHASE 3: Per-family evaluation + model save")
print(f"{'=' * 60}")

_NUM_CLASSES = 5
class_names = ['correct', 'substitution', 'insertion', 'deletion', 'HP_error']

# Prediction helper
def _predict_classes(booster, X, return_probs=False):
    """5-class prediction from booster."""
    dm = xgb.DMatrix(X) if not isinstance(X, xgb.DMatrix) else X
    probs = booster.predict(dm)
    if probs.ndim == 1:
        probs = probs.reshape(-1, _NUM_CLASSES)
    y_pred = probs.argmax(axis=1).astype(int)
    if return_probs:
        return y_pred, probs
    return y_pred


def _apply_calibration(raw_probs, calibrators):
    """Apply per-class isotonic calibration and renormalize."""
    cal_probs = np.column_stack([
        calibrators[c].transform(raw_probs[:, c])
        for c in range(raw_probs.shape[1])
    ])
    row_sums = cal_probs.sum(axis=1, keepdims=True)
    cal_probs = np.divide(cal_probs, row_sums,
                          where=row_sums > 0,
                          out=np.zeros_like(cal_probs))
    return cal_probs


# Save directory
errorsmith_save_dir = os.path.join(MODELS_OUT, 'errorsmith')
os.makedirs(errorsmith_save_dir, exist_ok=True)

# Aggregate accumulators for overall metrics
_all_y_val  = []
_all_y_pred = []
_all_chem   = []
_all_probs  = []
_all_family = []

# Per-family evaluation results
_family_cv_results = {}

for fam_name in family_results:
    fr = family_results[fam_name]
    fam_info = FAMILIES[fam_name]
    _booster = fr['booster']
    best_params = fr['best_params']

    if _booster is None:
        print(f"\n  " + chr(9888) + f" {fam_name}: no booster — skipping")
        continue

    print(f"\n{'─' * 60}")
    print(f"  Family: {fam_name}")
    print(f"{'─' * 60}")

    # ── Reload Phase 1 sample for evaluation ────────────────────
    ckpts = _family_ckpt_paths(fam_name)
    with open(ckpts['phase1'], 'rb') as f:
        _p1 = pickle.load(f)
    X_sample = _p1['X_sample']
    y_sample = _p1['y_sample']
    t_sample = _p1['t_sample']
    del _p1; gc.collect()

    # ── Booster diagnostics ─────────────────────────────────────
    _cfg = json.loads(_booster.save_config())
    _lp  = _cfg.get('learner', {}).get('learner_model_param', {})
    _obj = _cfg.get('learner', {}).get('objective', {}).get('name', '?')
    _nf  = _lp.get('num_feature', '?')
    _nr  = _booster.num_boosted_rounds()
    print(f"    Objective: {_obj}, features: {_nf}, rounds: {_nr}")

    # ── True hold-out (15%) ─────────────────────────────────────
    (X_cv, X_eval, y_cv, y_eval,
     t_cv, t_eval) = train_test_split(
        X_sample, y_sample, t_sample,
        test_size=EVAL_HOLDOUT_FRAC, random_state=SEED, stratify=y_sample
    )
    print(f"    Hold-out: {len(y_eval):,} | CV pool: {len(y_cv):,}")

    y_eval_pred, _eval_probs = _predict_classes(_booster, X_eval, return_probs=True)
    _eval_acc = accuracy_score(y_eval, y_eval_pred)
    _eval_f1m = f1_score(y_eval, y_eval_pred, average='macro')
    print(f"    Hold-out: Acc={_eval_acc:.4f}  F1-macro={_eval_f1m:.4f}")

    # Per-chemistry within family
    for _cc in sorted(set(t_eval.astype(int))):
        _m = t_eval.astype(int) == _cc
        _cn = CHEMISTRY_NAMES.get(_cc, f'unk_{_cc}')
        _na = _m.sum()
        _a  = accuracy_score(y_eval[_m], y_eval_pred[_m])
        _fm = f1_score(y_eval[_m], y_eval_pred[_m], average='macro', zero_division=0)
        print(f"      {_cn:<35s} n={_na:>6,}  Acc={_a:.4f}  F1={_fm:.4f}")

    # ── 5-fold CV ───────────────────────────────────────────────
    skf = StratifiedKFold(n_splits=ES_N_CV_FOLDS, shuffle=True, random_state=SEED)
    cv_acc, cv_f1 = [], []
    cv_y_val, cv_y_pred, cv_chem, cv_probs_list = [], [], [], []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_sample, y_sample)):
        X_va = X_sample[va_idx]
        y_va = y_sample[va_idx]
        t_va = t_sample[va_idx]
        y_vp, vp_probs = _predict_classes(_booster, X_va, return_probs=True)

        acc = accuracy_score(y_va, y_vp)
        f1  = f1_score(y_va, y_vp, average='macro')
        cv_acc.append(acc); cv_f1.append(f1)
        cv_y_val.append(y_va.copy())
        cv_y_pred.append(y_vp.copy())
        cv_chem.append(t_va.copy())
        cv_probs_list.append(vp_probs.copy())

        print(f"    Fold {fold+1}: Acc={acc:.4f}  F1-macro={f1:.4f}")

    print(f"    CV Mean: Acc={np.mean(cv_acc):.4f}{chr(177)}{np.std(cv_acc):.4f}  "
          f"F1={np.mean(cv_f1):.4f}{chr(177)}{np.std(cv_f1):.4f}")

    # Pool OOF predictions
    fam_y_val  = np.concatenate(cv_y_val)
    fam_y_pred = np.concatenate(cv_y_pred)
    fam_chem   = np.concatenate(cv_chem)
    fam_probs  = np.vstack(cv_probs_list)

    # ── Fit per-class isotonic calibration on OOF predictions ───
    _calibrators = {}
    for _cls_idx in range(_NUM_CLASSES):
        _ir = IsotonicRegression(out_of_bounds='clip')
        _ir.fit(fam_probs[:, _cls_idx],
                (fam_y_val == _cls_idx).astype(float))
        _calibrators[_cls_idx] = _ir
    fr['calibrators'] = _calibrators

    # Evaluate calibrated predictions on hold-out (unbiased)
    _holdout_cal_probs = _apply_calibration(_eval_probs, _calibrators)
    y_eval_cal = _holdout_cal_probs.argmax(axis=1).astype(int)
    _cal_acc = accuracy_score(y_eval, y_eval_cal)
    _cal_f1m = f1_score(y_eval, y_eval_cal, average='macro')
    print(f"    Calibrated hold-out: Acc={_cal_acc:.4f}  F1-macro={_cal_f1m:.4f}")

    # Append to global aggregates
    _all_y_val.append(fam_y_val)
    _all_y_pred.append(fam_y_pred)
    _all_chem.append(fam_chem)
    _all_probs.append(fam_probs)
    _all_family.append(np.full(len(fam_y_val), fam_name, dtype=object))

    # Classification report
    print(f"\n    Classification report ({fam_name}):")
    print(classification_report(fam_y_val, fam_y_pred,
                                target_names=class_names, digits=3))

    _family_cv_results[fam_name] = {
        'cv_acc_mean': float(np.mean(cv_acc)),
        'cv_acc_std':  float(np.std(cv_acc)),
        'cv_f1_mean':  float(np.mean(cv_f1)),
        'cv_f1_std':   float(np.std(cv_f1)),
        'holdout_acc': float(_eval_acc),
        'holdout_f1':  float(_eval_f1m),
        'holdout_acc_cal': float(_cal_acc),
        'holdout_f1_cal':  float(_cal_f1m),
        'n_sample':    len(y_sample),
        'total_trees': fr.get('total_trees', 0),
    }

    # ── Save family model ───────────────────────────────────────
    bst_path = os.path.join(errorsmith_save_dir, f'error_classifier_{fam_name}.ubj')
    _booster.save_model(bst_path)
    print(f"    " + chr(10003) + f" Saved: {bst_path}")

    pkl_path = os.path.join(errorsmith_save_dir, f'error_classifier_{fam_name}.pkl')
    _wrapper = xgb.XGBClassifier(**best_params)
    _wrapper._Booster = _booster
    try:
        _wrapper.classes_ = np.arange(_NUM_CLASSES)
        _wrapper.n_classes_ = _NUM_CLASSES
    except (AttributeError, TypeError):
        pass
    with open(pkl_path, 'wb') as f:
        pickle.dump(_wrapper, f)
    print(f"    " + chr(10003) + f" Saved: {pkl_path}")

    # Save calibrators
    _cal_path = os.path.join(errorsmith_save_dir, f'calibrators_{fam_name}.pkl')
    with open(_cal_path, 'wb') as f:
        pickle.dump(_calibrators, f)
    print(f"    " + chr(10003) + f" Saved: {_cal_path}")

    del X_sample, y_sample, t_sample, X_eval, y_eval, t_eval
    del X_cv, y_cv, t_cv
    gc.collect()

# ── Aggregate all families ──────────────────────────────────────────
_all_y_val  = np.concatenate(_all_y_val)
_all_y_pred = np.concatenate(_all_y_pred)
_all_chem   = np.concatenate(_all_chem)
_all_probs  = np.vstack(_all_probs)
_all_family = np.concatenate(_all_family)

_overall_acc = accuracy_score(_all_y_val, _all_y_pred)
_overall_f1  = f1_score(_all_y_val, _all_y_pred, average='macro')

print(f"\n{'=' * 60}")
print(f"  OVERALL (all families pooled):")
print(f"  Accuracy: {_overall_acc:.4f}  F1-macro: {_overall_f1:.4f}")
print(f"{'=' * 60}")
print(classification_report(_all_y_val, _all_y_pred,
                            target_names=class_names, digits=3))


# ── Save family routing map ─────────────────────────────────────────
routing = {
    'chemistry_to_family': {str(k): v for k, v in CHEMISTRY_TO_FAMILY.items()},
    'families': {
        fn: {
            'desc': FAMILIES[fn]['desc'],
            'codes': FAMILIES[fn]['codes'],
            'model_file': f'error_classifier_{fn}.ubj',
            'pkl_file': f'error_classifier_{fn}.pkl',
            'calibrator_file': f'calibrators_{fn}.pkl',
            'cv_f1_macro': _family_cv_results[fn]['cv_f1_mean'],
        }
        for fn in family_results if fn in _family_cv_results
    },
    'features': FAMILY_FEATURES,
    'n_features': len(FAMILY_FEATURES),
}
routing_path = os.path.join(errorsmith_save_dir, 'family_routing.json')
with open(routing_path, 'w') as f:
    json.dump(routing, f, indent=2, default=str)
print(f"\n" + chr(10003) + f" Saved: {routing_path}")


# ── Detect references ───────────────────────────────────────────────
_csv_name = str(errorsmith_csv).lower()
_refs_used = []
if 'hg002' in _csv_name:
    _refs_used.append('HG002v1.0.1')
if 'chm13' in _csv_name:
    _refs_used.append('CHM13v2.0')
_ref_str = ' + '.join(_refs_used) if _refs_used else 'dual-reference (HG002v1.0.1 + CHM13v2.0)'

# ── errorsmith_results (for downstream cells) ──────────────────────
errorsmith_results = {
    'architecture': 'per_chemistry_ensemble',
    'n_families': len(family_results),
    'families': _family_cv_results,
    'overall_accuracy': float(_overall_acc),
    'overall_f1_macro': float(_overall_f1),
    'n_samples_total': n_raw,
    'n_features': len(FAMILY_FEATURES),
    'feature_names': FAMILY_FEATURES,
    'reference': _ref_str,
    'chemistries_used': [CHEMISTRY_NAMES.get(int(c), str(c)) for c in sorted(_unique_chems)],
    'routing': routing,
}
with open(os.path.join(errorsmith_save_dir, 'training_results.json'), 'w') as f:
    json.dump(errorsmith_results, f, indent=2, default=str)

# ── Copy to GDrive ──────────────────────────────────────────────────
import shutil
GDRIVE_MODEL_DIR = os.path.join(GDRIVE_CHECKPOINT_DIR, 'final_model')
os.makedirs(GDRIVE_MODEL_DIR, exist_ok=True)
for _fn in os.listdir(errorsmith_save_dir):
    _src_f = os.path.join(errorsmith_save_dir, _fn)
    if os.path.isfile(_src_f):
        shutil.copy2(_src_f, os.path.join(GDRIVE_MODEL_DIR, _fn))
print(chr(128190) + f" Final models saved to GDrive: {GDRIVE_MODEL_DIR}")

print(f"\n{'=' * 60}")
print(f"  ErrorSmith training complete! (Per-chemistry ensemble)")
for fn in family_results:
    fr = family_results[fn]
    cv = _family_cv_results.get(fn, {})
    print(f"    {fn:20s}  trees={fr.get('total_trees', '?'):>5}  "
          f"F1={cv.get('cv_f1_mean', 0):.4f}")
print(f"  Overall F1-macro: {_overall_f1:.4f}")
print(f"  Features: {len(FAMILY_FEATURES)} per model")
print(f"  Reference: {_ref_str}")
print(f"{'=' * 60}")

PHASE 3: Per-family evaluation + model save

────────────────────────────────────────────────────────────
  Family: ont_r9
────────────────────────────────────────────────────────────
    Objective: multi:softprob, features: 34, rounds: 627
    Hold-out: 150,000 | CV pool: 850,000
    Hold-out: Acc=0.7807  F1-macro=0.7799
      ont_lsk110_r941                     n=75,191  Acc=0.7498  F1=0.7475
      ont_ulk001_r941                     n=74,809  Acc=0.8117  F1=0.8111
    Fold 1: Acc=0.7799  F1-macro=0.7791
    Fold 2: Acc=0.7810  F1-macro=0.7802
    Fold 3: Acc=0.7812  F1-macro=0.7803
    Fold 4: Acc=0.7808  F1-macro=0.7798
    Fold 5: Acc=0.7806  F1-macro=0.7796
    CV Mean: Acc=0.7807±0.0004  F1=0.7798±0.0005
    Calibrated hold-out: Acc=0.7878  F1-macro=0.7887

    Classification report (ont_r9):
              precision    recall  f1-score   support

     correct      0.693     0.847     0.762    200000
substitution      0.616     0.643     0.629    200000
   insertion      0.686   

## 6. Diagnostic Plots

Confusion matrix, per-class metrics, Optuna HP search history, and class balance visualization.
All plots use the **aggregated out-of-fold predictions** from the 5-fold CV (no data leakage).

In [14]:
# ── 6a. Confusion Matrix (normalized) ────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import shutil as _shutil_plot
from sklearn.metrics import confusion_matrix

# ── GDrive export directory for all diagnostic plots ────────────────
GDRIVE_PLOTS_DIR = os.path.join(GDRIVE_DIR, 'errorsmith_plots')
os.makedirs(GDRIVE_PLOTS_DIR, exist_ok=True)

def _save_plot(fig_path):
    """Save plot locally + copy to GDrive."""
    _shutil_plot.copy2(fig_path, os.path.join(GDRIVE_PLOTS_DIR, os.path.basename(fig_path)))

class_names = ['correct', 'substitution', 'insertion', 'deletion', 'HP_error']

cm_raw = confusion_matrix(_all_y_val, _all_y_pred)
cm_norm = cm_raw.astype(float) / cm_raw.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Confusion Matrix — ErrorSmith (Aggregated 5-Fold CV)', fontsize=14, fontweight='bold')

# Normalized
im0 = axes[0].imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
axes[0].set_title('Normalized (recall per class)')
for i in range(len(class_names)):
    for j in range(len(class_names)):
        val = cm_norm[i, j]
        color = 'white' if val > 0.5 else 'black'
        axes[0].text(j, i, f'{val:.2%}', ha='center', va='center', color=color, fontsize=10)
axes[0].set_xticks(range(len(class_names)))
axes[0].set_xticklabels(class_names, rotation=45, ha='right', fontsize=9)
axes[0].set_yticks(range(len(class_names)))
axes[0].set_yticklabels(class_names, fontsize=9)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')
plt.colorbar(im0, ax=axes[0], fraction=0.046)

# Raw counts
im1 = axes[1].imshow(cm_raw, cmap='Oranges')
axes[1].set_title('Raw counts')
for i in range(len(class_names)):
    for j in range(len(class_names)):
        val = cm_raw[i, j]
        max_val = cm_raw.max()
        color = 'white' if val > max_val * 0.5 else 'black'
        axes[1].text(j, i, f'{val:,}', ha='center', va='center', color=color, fontsize=9)
axes[1].set_xticks(range(len(class_names)))
axes[1].set_xticklabels(class_names, rotation=45, ha='right', fontsize=9)
axes[1].set_yticks(range(len(class_names)))
axes[1].set_yticklabels(class_names, fontsize=9)
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')
plt.colorbar(im1, ax=axes[1], fraction=0.046)

plt.tight_layout(rect=[0, 0, 1, 0.94])
_fig_path = os.path.join(MODELS_OUT, 'errorsmith_confusion_matrix.png')
plt.savefig(_fig_path, dpi=150, bbox_inches='tight')
_save_plot(_fig_path)
plt.show()
print("✓ Confusion matrix saved → GDrive")

✓ Confusion matrix saved → GDrive


In [15]:
# ── 6b. Per-Class Precision / Recall / F1 ────────────────────────────
from sklearn.metrics import classification_report

report = classification_report(
    _all_y_val, _all_y_pred,
    target_names=class_names,
    output_dict=True,
    zero_division=0,
)

metrics = ['precision', 'recall', 'f1-score']
x_pos = np.arange(len(class_names))
width = 0.25
colors = ['#2196F3', '#4CAF50', '#FF9800']

fig, ax = plt.subplots(figsize=(12, 5))
for i, m in enumerate(metrics):
    vals = [report[cn][m] for cn in class_names]
    bars = ax.bar(x_pos + i * width, vals, width, label=m.title(), color=colors[i], edgecolor='white')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f'{v:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x_pos + width)
ax.set_xticklabels(class_names, fontsize=10)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.12)
ax.set_title('Per-Class Metrics — Aggregated 5-Fold CV', fontweight='bold')
ax.legend(loc='lower right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
_fig_path = os.path.join(MODELS_OUT, 'errorsmith_per_class_metrics.png')
plt.savefig(_fig_path, dpi=150, bbox_inches='tight')
_save_plot(_fig_path)
plt.show()

# Macro / Weighted averages
print(f"\n{'Metric':<20} {'Macro':>8} {'Weighted':>8}")
print("-" * 38)
for m in metrics:
    print(f"{m.title():<20} {report['macro avg'][m]:>8.4f} {report['weighted avg'][m]:>8.4f}")
print(f"{'Support':<20} {int(report['macro avg']['support']):>8,}")


Metric                  Macro Weighted
--------------------------------------
Precision              0.8604   0.8543
Recall                 0.8543   0.8522
F1-Score               0.8552   0.8511
Support              6,237,832


In [16]:
# ── 6c. Training Progress (per-family) ────────────────────────────────
n_fams = len(family_results)
fig, axes = plt.subplots(n_fams, 3, figsize=(20, 5 * n_fams))
if n_fams == 1:
    axes = axes.reshape(1, -1)
fig.suptitle('Training Progress — Per-Chemistry Ensemble', fontsize=14, fontweight='bold')

for fi, fam_name in enumerate(family_results):
    fr = family_results[fam_name]

    # Panel 1: Optuna trial scores
    _study = fr.get('study', None)
    if _study is not None:
        trials_df = _study.trials_dataframe().sort_values('number')
        best_so_far = trials_df['value'].cummax()
        axes[fi, 0].scatter(trials_df['number'], trials_df['value'],
                            alpha=0.5, s=30, c='steelblue', label='Trial F1')
        axes[fi, 0].plot(trials_df['number'], best_so_far, 'r-', linewidth=2, label='Best')
        axes[fi, 0].axhline(y=_study.best_value, color='green', linestyle='--', alpha=0.5)
        axes[fi, 0].legend(fontsize=7)
    else:
        axes[fi, 0].text(0.5, 0.5, f'Loaded from checkpoint\nF1={fr["p1_f1"]:.4f}',
                         ha='center', va='center', transform=axes[fi, 0].transAxes, fontsize=11)
    axes[fi, 0].set_title(f'{fam_name}: Optuna HP Search', fontsize=10)
    axes[fi, 0].grid(alpha=0.3)

    # Panel 2: Incremental training F1
    _rf = fr.get('round_f1s', [])
    if _rf:
        rounds = range(1, len(_rf) + 1)
        axes[fi, 1].plot(rounds, _rf, 'o-', color='#FF5722', markersize=5, linewidth=2)
        axes[fi, 1].fill_between(rounds, _rf, alpha=0.15, color='#FF5722')
        axes[fi, 1].set_xlabel('Round')
        axes[fi, 1].set_ylabel('Train-F1')
        axes[fi, 1].set_title(f'{fam_name}: Incremental Training ({len(_rf)} rounds)', fontsize=10)
    else:
        axes[fi, 1].text(0.5, 0.5, 'No rounds', ha='center', va='center',
                         transform=axes[fi, 1].transAxes)
        axes[fi, 1].set_title(f'{fam_name}: Incremental Training', fontsize=10)
    axes[fi, 1].grid(alpha=0.3)

    # Panel 3: Best HP table
    axes[fi, 2].axis('off')
    bp = fr.get('best_params', {})
    _bp_rows = [[k, f'{v:.4f}' if isinstance(v, float) else str(v)]
                for k, v in bp.items()
                if k not in ('tree_method', 'device', 'random_state', 'seed')]
    if _bp_rows:
        table = axes[fi, 2].table(cellText=_bp_rows,
                                   colLabels=['Parameter', 'Value'],
                                   cellLoc='center', loc='center')
        table.auto_set_font_size(False)
        table.set_fontsize(8)
        table.scale(1, 1.2)
    axes[fi, 2].set_title(f'{fam_name}: Best HPs', fontsize=10, pad=20)

plt.tight_layout(rect=[0, 0, 1, 0.95])
_fig_path = os.path.join(MODELS_OUT, 'errorsmith_training_progress.png')
plt.savefig(_fig_path, dpi=150, bbox_inches='tight')
_save_plot(_fig_path)
plt.show()

# Summary
for fam_name in family_results:
    fr = family_results[fam_name]
    _rf = fr.get('round_f1s', [])
    print(f"  {fam_name}: F1-HP={fr['p1_f1']:.4f}, "
          f"rounds={len(_rf)}, "
          f"final-train-F1={_rf[-1]:.4f}" if _rf else f"  {fam_name}: F1-HP={fr['p1_f1']:.4f}")

  ont_r9: F1-HP=0.8806, rounds=11, final-train-F1=0.9488
  ont_lsk114_r1041: F1-HP=0.8460, rounds=6, final-train-F1=0.9781
  ont_ulk114_r1041: F1-HP=0.9040, rounds=12, final-train-F1=0.9115
  ont_hiacc: F1-HP=0.9036, rounds=11, final-train-F1=0.9953
  lr_accurate: F1-HP=0.9347, rounds=12, final-train-F1=0.9809
  short_read_mixed: F1-HP=0.9807, rounds=12, final-train-F1=0.9776
  short_read_sbb: F1-HP=0.9900, rounds=21, final-train-F1=1.0000


In [17]:
# ── 6d. Class + Chemistry Distribution ───────────────────────────────
from collections import Counter as _Counter24

# Raw distributions — from Phase 1 full-CSV scan (_chem_class_counts)
_raw_class_dist = _Counter24()
_raw_chem_dist  = _Counter24()
for _cc24, _cls_counts24 in _chem_class_counts.items():
    for _cls24, _cnt24 in _cls_counts24.items():
        _raw_class_dist[int(_cls24)] += _cnt24
    _raw_chem_dist[int(_cc24)] += sum(_cls_counts24.values())

# Balanced distributions — from Phase 3 OOF evaluation pool
# (reflects class-balanced, family-routed training proportions)
_balanced_class_dist = _Counter24(int(v) for v in _all_y_val)
_balanced_chem_dist  = _Counter24(int(v) for v in _all_chem)

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('Data Distribution — Before & After Resampling', fontsize=14, fontweight='bold')

_palette = ['#4CAF50', '#FF5722', '#2196F3', '#9C27B0', '#FF9800']

# ── Row 1: Class distribution ────────────────────────────────────────
# Raw
raw_labels = sorted(_raw_class_dist.keys())
raw_vals   = [_raw_class_dist[k] for k in raw_labels]
raw_names  = [class_names[k] if k < len(class_names) else f'code_{k}' for k in raw_labels]
bars0 = axes[0, 0].bar(raw_names, raw_vals, color=[_palette[k % len(_palette)] for k in raw_labels], edgecolor='white')
axes[0, 0].set_title('Classes — Raw (after stratified load)')
axes[0, 0].set_ylabel('Samples')
axes[0, 0].tick_params(axis='x', rotation=35)
for bar, v in zip(bars0, raw_vals):
    axes[0, 0].text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                    f'{v:,}', ha='center', va='bottom', fontsize=7)
axes[0, 0].grid(axis='y', alpha=0.3)

# Balanced
bal_labels = sorted(_balanced_class_dist.keys())
bal_vals   = [_balanced_class_dist[k] for k in bal_labels]
bal_names  = [class_names[k] if k < len(class_names) else f'code_{k}' for k in bal_labels]
bars1 = axes[0, 1].bar(bal_names, bal_vals, color=[_palette[k % len(_palette)] for k in bal_labels], edgecolor='white')
axes[0, 1].set_title('Classes — After chem×class resampling')
axes[0, 1].set_ylabel('Samples')
axes[0, 1].tick_params(axis='x', rotation=35)
for bar, v in zip(bars1, bal_vals):
    axes[0, 1].text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                    f'{v:,}', ha='center', va='bottom', fontsize=7)
axes[0, 1].grid(axis='y', alpha=0.3)

# ── Row 2: Chemistry distribution ────────────────────────────────────
_chem_palette = plt.cm.tab20(np.linspace(0, 1, max(len(_raw_chem_dist), 1)))

# Raw chemistry
raw_ch_labels = sorted(_raw_chem_dist.keys())
raw_ch_vals   = [_raw_chem_dist[k] for k in raw_ch_labels]
raw_ch_names  = [CHEMISTRY_NAMES.get(int(k), f'code_{k}')[:20] for k in raw_ch_labels]
bars2 = axes[1, 0].barh(range(len(raw_ch_names)), raw_ch_vals,
                         color=_chem_palette[:len(raw_ch_names)], edgecolor='white')
axes[1, 0].set_yticks(range(len(raw_ch_names)))
axes[1, 0].set_yticklabels(raw_ch_names, fontsize=8)
axes[1, 0].set_title('Chemistries — Raw (after stratified load)')
axes[1, 0].set_xlabel('Samples')
for i, (bar, v) in enumerate(zip(bars2, raw_ch_vals)):
    axes[1, 0].text(bar.get_width() + max(raw_ch_vals) * 0.01, i,
                    f'{v:,}', va='center', fontsize=7)
axes[1, 0].grid(axis='x', alpha=0.3)

# Balanced chemistry
bal_ch_labels = sorted(_balanced_chem_dist.keys())
bal_ch_vals   = [_balanced_chem_dist[k] for k in bal_ch_labels]
bal_ch_names  = [CHEMISTRY_NAMES.get(int(k), f'code_{k}')[:20] for k in bal_ch_labels]
bars3 = axes[1, 1].barh(range(len(bal_ch_names)), bal_ch_vals,
                         color=_chem_palette[:len(bal_ch_names)], edgecolor='white')
axes[1, 1].set_yticks(range(len(bal_ch_names)))
axes[1, 1].set_yticklabels(bal_ch_names, fontsize=8)
axes[1, 1].set_title('Chemistries — After chem×class resampling')
axes[1, 1].set_xlabel('Samples')
for i, (bar, v) in enumerate(zip(bars3, bal_ch_vals)):
    axes[1, 1].text(bar.get_width() + max(bal_ch_vals) * 0.01, i,
                    f'{v:,}', va='center', fontsize=7)
axes[1, 1].grid(axis='x', alpha=0.3)

plt.tight_layout(rect=[0, 0, 1, 0.95])
_fig_path = os.path.join(MODELS_OUT, 'errorsmith_data_distribution.png')
plt.savefig(_fig_path, dpi=150, bbox_inches='tight')
_save_plot(_fig_path)
plt.show()

# Imbalance ratios
max_raw = max(raw_vals) if raw_vals else 0
min_raw = min(raw_vals) if raw_vals else 1
print(f"Class imbalance — raw: {max_raw / max(min_raw, 1):.1f}x, balanced: {max(bal_vals) / max(min(bal_vals), 1):.1f}x")
print(f"Chemistry — {len(raw_ch_labels)} represented (raw), {len(bal_ch_labels)} represented (balanced)")

Class imbalance — raw: 15.8x, balanced: 1.4x
Chemistry — 14 represented (raw), 14 represented (balanced)


In [18]:
# ── 6e. Feature Importance (per-family) ───────────────────────────────
n_fams = len(family_results)
feature_names = FAMILY_FEATURES
n_features = len(feature_names)

fig, axes = plt.subplots(1, n_fams, figsize=(8 * n_fams, max(6, n_features * 0.35)))
if n_fams == 1:
    axes = [axes]
fig.suptitle('Feature Importance — Per-Chemistry Ensemble', fontsize=14, fontweight='bold')

for fi, fam_name in enumerate(family_results):
    fr = family_results[fam_name]
    _booster = fr.get('booster', None)
    ax = axes[fi]

    if _booster is None:
        ax.text(0.5, 0.5, 'No booster', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(fam_name)
        continue

    _imp_dict = _booster.get_score(importance_type='gain')
    importances = np.zeros(n_features, dtype=np.float32)
    for k, v in _imp_dict.items():
        idx = int(k.replace('f', ''))
        if idx < n_features:
            importances[idx] = v
    if importances.sum() > 0:
        importances = importances / importances.sum()

    idx = np.argsort(importances)
    sorted_names = [feature_names[i] for i in idx]
    sorted_vals  = importances[idx]

    bars = ax.barh(range(n_features), sorted_vals, color='steelblue', edgecolor='white')
    # Highlight prior features
    for i, name in enumerate(sorted_names):
        if name.startswith('chemistry_'):
            bars[i].set_color('#FF9800')

    ax.set_yticks(range(n_features))
    ax.set_yticklabels(sorted_names, fontsize=8)
    ax.set_xlabel('Normalized gain')
    ax.set_title(fam_name, fontweight='bold', fontsize=11)
    ax.grid(axis='x', alpha=0.3)

    for i, v in enumerate(sorted_vals):
        ax.text(v + max(sorted_vals) * 0.01, i, f'{v:.3f}', va='center', fontsize=6)

plt.tight_layout(rect=[0, 0, 1, 0.94])
_fig_path = os.path.join(MODELS_OUT, 'errorsmith_feature_importance.png')
plt.savefig(_fig_path, dpi=150, bbox_inches='tight')
_save_plot(_fig_path)
plt.show()

# Top features per family
for fam_name in family_results:
    fr = family_results[fam_name]
    _booster = fr.get('booster', None)
    if _booster is None:
        continue
    _imp_dict = _booster.get_score(importance_type='gain')
    importances = np.zeros(n_features, dtype=np.float32)
    for k, v in _imp_dict.items():
        idx = int(k.replace('f', ''))
        if idx < n_features:
            importances[idx] = v
    if importances.sum() > 0:
        importances /= importances.sum()
    top5 = np.argsort(importances)[-5:][::-1]
    print(f"\n  {fam_name} — Top 5:")
    for i in top5:
        print(f"    {feature_names[i]:<30s} {importances[i]:.4f}")


  ont_r9 — Top 5:
    base_quality                   0.1488
    ref_homopolymer_length         0.1384
    homopolymer_length             0.1161
    homopolymer_base               0.0674
    ref_repeat_flag                0.0459

  ont_lsk114_r1041 — Top 5:
    base_quality                   0.1873
    homopolymer_length             0.1792
    ref_homopolymer_length         0.1688
    homopolymer_base               0.1245
    kmer_solid_ratio               0.0539

  ont_ulk114_r1041 — Top 5:
    ref_homopolymer_length         0.1432
    base_quality                   0.1220
    homopolymer_length             0.1082
    homopolymer_base               0.0871
    kmer_log_ratio                 0.0710

  ont_hiacc — Top 5:
    ref_homopolymer_length         0.2147
    homopolymer_base               0.1231
    base_quality                   0.1137
    kmer_min_freq                  0.0918
    kmer_solid_ratio               0.0899

  lr_accurate — Top 5:
    kmer_min_freq                  0.

In [19]:
# ── 6f. Error Distribution Predictions by Chemistry ──────────────────
#
# Uses REAL ground-truth error rates from the full CSV scan (Pass 1),
# NOT the class-balanced training sample.
# Predicted rates come from model inference on RAW (unbalanced) data.
# Confusion matrices still use CV results (valid for accuracy measurement).
#
# ⚠ IMPORTANT: The training CSV subsamples CORRECT bases to only 2%
#   (CORRECT_SUBSAMPLE_RATE = 0.02) to keep the dataset manageable.
#   Raw CSV counts therefore vastly over-represent error classes.
#   We correct for this below by inflating correct-base counts by
#   1/CORRECT_SUBSAMPLE_RATE before computing true error rates.

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import confusion_matrix
from collections import defaultdict, Counter
import numpy as np
import pandas as pd
import gc

class_names = ['correct', 'substitution', 'insertion', 'deletion', 'HP_error']
_class_colors = ['#4CAF50', '#FF5722', '#2196F3', '#9C27B0', '#FF9800']

# ── Correct-base subsampling rate (must match the generator) ─────────
# The generator keeps only this fraction of "correct" bases in the CSV.
CORRECT_SUBSAMPLE_RATE = 0.02

# ═══════════════════════════════════════════════════════════════════════
#  STEP 1: Get real ground-truth error rates from _chem_class_counts
# ═══════════════════════════════════════════════════════════════════════
# _chem_class_counts was built in Phase 1 Pass 1 by scanning ALL rows
# of the CSV *without* class balancing. If unavailable (old checkpoint
# that didn't save it), rescan now.

if '_chem_class_counts' not in dir() or _chem_class_counts is None:
    print("  ⚠ _chem_class_counts not available — rescanning CSV for true rates...")
    _chem_class_counts = defaultdict(Counter)
    _class_totals = Counter()
    for _mchunk in pd.read_csv(errorsmith_csv,
                                usecols=['technology_encoded', ES_LABEL],
                                chunksize=2_000_000):
        for (tech, cls), cnt in _mchunk.groupby(
                ['technology_encoded', ES_LABEL]).size().items():
            chem = _remap_tech_code(tech)
            _chem_class_counts[chem][cls] += cnt
            _class_totals[cls] += cnt
    del _mchunk; gc.collect()
    print(f"  ✓ Rescanned {sum(_class_totals.values()):,} rows")
else:
    print(f"  ✓ Using saved _chem_class_counts ({sum(_class_totals.values()):,} rows)")

# Build true-rate matrix from raw counts — CORRECTED for subsampling
# The CSV kept only 2% of correct bases, so we multiply the correct
# count by 1/CORRECT_SUBSAMPLE_RATE to recover approximate true counts.
_uniq_chems = sorted(set(int(c) for c in _chem_class_counts.keys()))
_chem_labels = [CHEMISTRY_NAMES.get(c, f'unk_{c}') for c in _uniq_chems]
_n_chems = len(_uniq_chems)

_true_rate_matrix = np.zeros((_n_chems, len(class_names)))
for ci, cc in enumerate(_uniq_chems):
    counts = _chem_class_counts[cc]
    # Correct for subsampling: inflate correct count by 1/rate
    csv_correct = counts.get(0, 0)
    corrected_correct = csv_correct / CORRECT_SUBSAMPLE_RATE
    total_errors = sum(counts.get(cls, 0) for cls in range(1, len(class_names)))
    corrected_total = corrected_correct + total_errors
    if corrected_total > 0:
        _true_rate_matrix[ci, 0] = corrected_correct / corrected_total
        for cls in range(1, len(class_names)):
            _true_rate_matrix[ci, cls] = counts.get(cls, 0) / corrected_total

print(f"\n  Ground-truth error rates (corrected for {CORRECT_SUBSAMPLE_RATE:.0%} correct-base subsampling):")
print(f"  {'Chemistry':<35s}  {'Error%':>8s}  {'Sub%':>7s}  {'Ins%':>7s}  {'Del%':>7s}  {'HP%':>7s}")
print(f"  {'─'*35}  {'─'*8}  {'─'*7}  {'─'*7}  {'─'*7}  {'─'*7}")
for ci, cc in enumerate(_uniq_chems):
    name = CHEMISTRY_NAMES.get(cc, f'unk_{cc}')
    r = _true_rate_matrix[ci]
    err = 1.0 - r[0]
    print(f"  {name:<35s}  {err:>7.3%}  {r[1]:>6.3%}  {r[2]:>6.3%}  {r[3]:>6.3%}  {r[4]:>6.3%}")


# ═══════════════════════════════════════════════════════════════════════
#  STEP 2: Raw-data inference — predict on UNBALANCED data
#
#  Stream a raw sample per chemistry (no class balancing) and predict.
#  This shows the error-rate distribution the model WOULD output on
#  real-world data from each platform.
#
#  Same subsampling correction is applied to predicted class counts
#  so that ground truth and predicted rates are directly comparable
#  and reflect approximate real-world error rates.
# ═══════════════════════════════════════════════════════════════════════
_RAW_SAMPLE_PER_CHEM = 50_000
print(f"\n  Streaming raw (unbalanced) sample for inference "
      f"({_RAW_SAMPLE_PER_CHEM:,} rows/chem)...")

_raw_buf = defaultdict(list)          # chem → list of numpy arrays
_raw_sample_counts = Counter()        # chem → rows collected so far

for _rchunk in pd.read_csv(errorsmith_csv, usecols=_usecols, chunksize=2_000_000):
    _rchunk = _derive_features(_rchunk)
    _rtechs = _rchunk['technology_encoded'].values.astype(int)
    _arr = _rchunk[FAMILY_FEATURES].values

    for cc in _uniq_chems:
        if _raw_sample_counts[cc] >= _RAW_SAMPLE_PER_CHEM:
            continue
        mask = _rtechs == cc
        if mask.sum() == 0:
            continue
        need = _RAW_SAMPLE_PER_CHEM - _raw_sample_counts[cc]
        rows = _arr[mask][:need]
        _raw_buf[cc].append(rows)
        _raw_sample_counts[cc] += len(rows)

    # Early stop if all chemistries have enough
    if all(_raw_sample_counts[cc] >= _RAW_SAMPLE_PER_CHEM for cc in _uniq_chems):
        break

del _rchunk, _arr; gc.collect()

# Predict on each chemistry's raw sample — CORRECTED for subsampling
_pred_rate_matrix = np.zeros((_n_chems, len(class_names)))
print(f"\n  Predicted error rates (corrected for {CORRECT_SUBSAMPLE_RATE:.0%} correct-base subsampling):")
print(f"  {'Chemistry':<35s}  {'N':>7s}  {'Pred Err%':>10s}  {'Sub%':>7s}  {'Ins%':>7s}  {'Del%':>7s}  {'HP%':>7s}")
print(f"  {'─'*35}  {'─'*7}  {'─'*10}  {'─'*7}  {'─'*7}  {'─'*7}  {'─'*7}")

for ci, cc in enumerate(_uniq_chems):
    if cc not in _raw_buf or len(_raw_buf[cc]) == 0:
        continue
    X_raw = np.vstack(_raw_buf[cc])
    # Route to the correct family's booster
    _fam = CHEMISTRY_TO_FAMILY.get(cc, None)
    if _fam is None or _fam not in family_results or family_results[_fam].get('booster') is None:
        continue
    _fam_booster = family_results[_fam]['booster']
    # Recompute features for this family's feature set (add priors if missing)
    _raw_df = pd.DataFrame(X_raw, columns=FAMILY_FEATURES)
    if 'chemistry_error_rate' not in _raw_df.columns:
        _raw_df['chemistry_error_rate'] = CHEMISTRY_ERROR_RATES.get(cc, 0.01)
        _raw_df['chemistry_hp_error_rate'] = CHEMISTRY_HP_ERROR_RATES.get(cc, 0.01)
    X_fam = _raw_df[FAMILY_FEATURES].fillna(0).values.astype(np.float32)
    # Calibrated prediction (uses isotonic calibrators from Phase 3)
    _fam_cals = family_results.get(_fam, {}).get('calibrators', None)
    _, _inf_raw_probs = _predict_classes(_fam_booster, X_fam, return_probs=True)
    if _fam_cals:
        _inf_cal_p = _apply_calibration(_inf_raw_probs, _fam_cals)
        preds = _inf_cal_p.argmax(axis=1).astype(int)
    else:
        preds = _inf_raw_probs.argmax(axis=1).astype(int)
    # Correct for subsampling: the input data has 2% correct retention,
    # so predicted "correct" counts are deflated. Inflate by 1/rate.
    pred_correct = int((preds == 0).sum())
    corrected_pred_correct = pred_correct / CORRECT_SUBSAMPLE_RATE
    pred_error_counts = {cls: int((preds == cls).sum()) for cls in range(1, len(class_names))}
    total_pred_errors = sum(pred_error_counts.values())
    corrected_total = corrected_pred_correct + total_pred_errors
    if corrected_total > 0:
        _pred_rate_matrix[ci, 0] = corrected_pred_correct / corrected_total
        for cls in range(1, len(class_names)):
            _pred_rate_matrix[ci, cls] = pred_error_counts[cls] / corrected_total
    name = CHEMISTRY_NAMES.get(cc, f'unk_{cc}')
    r = _pred_rate_matrix[ci]
    pred_err = 1.0 - r[0]
    print(f"  {name:<35s}  {len(preds):>7,}  {pred_err:>9.3%}  {r[1]:>6.3%}  {r[2]:>6.3%}  {r[3]:>6.3%}  {r[4]:>6.3%}")

del _raw_buf; gc.collect()


# ═══════════════════════════════════════════════════════════════════════
#  PLOT 1: True vs Predicted error type distributions (grouped bars)
#  TRUE = corrected ground truth | PREDICTED = corrected inference
# ═══════════════════════════════════════════════════════════════════════
# ── Errors-only stacked bars (correct class excluded for readability) ──
# The "correct" class is 95–99.9% of all bases, making error bars
# invisible on a 0–1 scale. Here we show only error classes,
# renormalized so that the 4 error types sum to 1 per chemistry.
_err_class_names  = class_names[1:]   # sub, ins, del, HP
_err_class_colors = _class_colors[1:]

# Renormalize: for each chemistry, error columns / sum(error columns)
_true_err_only = _true_rate_matrix[:, 1:].copy()
_pred_err_only = _pred_rate_matrix[:, 1:].copy()
_true_err_sums = _true_err_only.sum(axis=1, keepdims=True)
_pred_err_sums = _pred_err_only.sum(axis=1, keepdims=True)
_true_err_norm = np.divide(_true_err_only, _true_err_sums,
                           where=_true_err_sums > 0,
                           out=np.zeros_like(_true_err_only))
_pred_err_norm = np.divide(_pred_err_only, _pred_err_sums,
                           where=_pred_err_sums > 0,
                           out=np.zeros_like(_pred_err_only))

fig, axes = plt.subplots(2, 2, figsize=(20, max(12, _n_chems * 0.9)))
fig.suptitle('Error Type Distribution — Ground Truth vs Prediction\n'
             f'(corrected for {CORRECT_SUBSAMPLE_RATE:.0%} correct-base subsampling)',
             fontsize=13, fontweight='bold')

# ── Top row: total error rate per chemistry (single bar) ────────────
for panel, (title, rate_matrix) in enumerate([
    ('Total error rate — Ground truth', _true_rate_matrix),
    ('Total error rate — Predicted', _pred_rate_matrix),
]):
    ax = axes[0, panel]
    err_rates = (1.0 - rate_matrix[:, 0]) * 100  # error %
    bars = ax.barh(range(_n_chems), err_rates,
                   color='#FF5722', edgecolor='white', alpha=0.85)
    for ci in range(_n_chems):
        ax.text(err_rates[ci] + max(err_rates) * 0.02, ci,
                f'{err_rates[ci]:.2f}%', va='center', fontsize=8)
    ax.set_yticks(range(_n_chems))
    ax.set_yticklabels(_chem_labels, fontsize=9)
    ax.set_xlabel('Error rate (%)')
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.grid(axis='x', alpha=0.2)

# ── Bottom row: error type breakdown (stacked, errors only) ─────────
for panel, (title, err_norm) in enumerate([
    ('Error type breakdown — Ground truth', _true_err_norm),
    ('Error type breakdown — Predicted', _pred_err_norm),
]):
    ax = axes[1, panel]
    left = np.zeros(_n_chems)
    for cls_idx in range(len(_err_class_names)):
        ax.barh(range(_n_chems), err_norm[:, cls_idx], left=left,
                color=_err_class_colors[cls_idx],
                label=_err_class_names[cls_idx],
                edgecolor='white', linewidth=0.5)
        for ci in range(_n_chems):
            pct = err_norm[ci, cls_idx]
            if pct > 0.06:
                ax.text(left[ci] + pct / 2, ci, f'{pct:.0%}',
                        ha='center', va='center', fontsize=7,
                        color='white' if pct > 0.25 else 'black',
                        fontweight='bold')
        left += err_norm[:, cls_idx]
    ax.set_yticks(range(_n_chems))
    ax.set_yticklabels(_chem_labels, fontsize=9)
    ax.set_xlabel('Proportion of errors (excl. correct)')
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.set_xlim(0, 1)
    ax.grid(axis='x', alpha=0.2)
    if panel == 0:
        ax.legend(loc='lower right', fontsize=8, framealpha=0.9)

plt.tight_layout(rect=[0, 0, 1, 0.92])
_fig_path = os.path.join(MODELS_OUT, 'errorsmith_error_distribution_by_chemistry.png')
plt.savefig(_fig_path, dpi=150, bbox_inches='tight')
_save_plot(_fig_path)
plt.show()
print("✓ Error distribution comparison saved → GDrive")


# ═══════════════════════════════════════════════════════════════════════
#  PLOT 2: Per-chemistry confusion heatmaps (small multiples)
#  These use the BALANCED CV data — valid for measuring model accuracy
# ═══════════════════════════════════════════════════════════════════════
_cv_uniq_chems = sorted(set(int(c) for c in _all_chem))
_cv_n_chems = len(_cv_uniq_chems)
_cols = min(4, _cv_n_chems)
_rows = (_cv_n_chems + _cols - 1) // _cols
fig, axes = plt.subplots(_rows, _cols, figsize=(_cols * 4.5, _rows * 4))
fig.suptitle('Per-Chemistry Confusion Matrices (normalized)\n'
             '(from balanced CV — measures classification accuracy, not error rates)',
             fontsize=13, fontweight='bold', y=1.01)

if _cv_n_chems == 1:
    axes = np.array([[axes]])
elif _rows == 1:
    axes = axes.reshape(1, -1)

for ci, cc in enumerate(_cv_uniq_chems):
    r, c = divmod(ci, _cols)
    ax = axes[r, c]
    mask = _all_chem == cc
    y_t = _all_y_val[mask].astype(int)
    y_p = _all_y_pred[mask].astype(int)
    n = mask.sum()

    cm = confusion_matrix(y_t, y_p, labels=range(len(class_names)))
    cm_n = cm.astype(float) / np.maximum(cm.sum(axis=1, keepdims=True), 1)

    im = ax.imshow(cm_n, cmap='Blues', vmin=0, vmax=1)
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            val = cm_n[i, j]
            if val > 0.005:
                color = 'white' if val > 0.5 else 'black'
                ax.text(j, i, f'{val:.0%}', ha='center', va='center',
                        color=color, fontsize=7)

    cname = CHEMISTRY_NAMES.get(cc, f'unk_{cc}')
    ax.set_title(f'{cname}\n(n={n:,})', fontsize=9, fontweight='bold')
    ax.set_xticks(range(len(class_names)))
    ax.set_xticklabels([cn[:4] for cn in class_names], fontsize=7, rotation=45)
    ax.set_yticks(range(len(class_names)))
    ax.set_yticklabels([cn[:4] for cn in class_names], fontsize=7)

# Hide unused subplots
for ci in range(_cv_n_chems, _rows * _cols):
    r, c = divmod(ci, _cols)
    axes[r, c].set_visible(False)

plt.tight_layout()
_fig_path = os.path.join(MODELS_OUT, 'errorsmith_per_chemistry_confusion.png')
plt.savefig(_fig_path, dpi=150, bbox_inches='tight')
_save_plot(_fig_path)
plt.show()
print("✓ Per-chemistry confusion matrices saved → GDrive")


# ═══════════════════════════════════════════════════════════════════════
#  PLOT 3: Error rate profiles — line plot comparing all platforms
#  Left: ground-truth rates (corrected)
#  Right: predicted rates (corrected)
# ═══════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Error Rate Profiles Across Platforms\n'
             f'(corrected for {CORRECT_SUBSAMPLE_RATE:.0%} correct-base subsampling)',
             fontsize=13, fontweight='bold')

# Color-code by technology family
_family_colors = {}
for cc in _uniq_chems:
    name = CHEMISTRY_NAMES.get(cc, '')
    if 'ont' in name:
        _family_colors[cc] = plt.cm.Reds(0.4 + 0.05 * list(_uniq_chems).index(cc))
    elif 'pacbio' in name:
        _family_colors[cc] = plt.cm.Blues(0.4 + 0.08 * list(_uniq_chems).index(cc))
    elif 'illumina' in name:
        _family_colors[cc] = plt.cm.Greens(0.6)
    elif 'element' in name:
        _family_colors[cc] = plt.cm.Purples(0.4 + 0.08 * list(_uniq_chems).index(cc))
    else:
        _family_colors[cc] = 'gray'

# Plot error classes only (correct ≈ 95–99.9% crushes the y-axis)
_err_names = class_names[1:]
for panel, (title, rate_matrix) in enumerate([
    ('Ground-truth error rates (corrected)', _true_rate_matrix),
    ('Predicted error rates (corrected)', _pred_rate_matrix),
]):
    ax = axes[panel]
    x = np.arange(len(_err_names))

    for ci, cc in enumerate(_uniq_chems):
        rates = rate_matrix[ci, 1:]  # skip class 0 (correct)
        if rates.sum() == 0:
            continue
        cname = CHEMISTRY_NAMES.get(cc, f'unk_{cc}')
        short = cname.replace('ont_', 'ONT ').replace('pacbio_', 'PB ') \
                      .replace('illumina_', 'Illum ').replace('element_', 'Elem ')
        ax.plot(x, rates, 'o-', label=short, color=_family_colors[cc],
                markersize=5, linewidth=1.5, alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(_err_names, fontsize=10, rotation=20)
    ax.set_ylabel('Error rate (proportion of bases)')
    ax.set_yscale('log')
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.grid(alpha=0.3, which='both')

# Single legend for both panels
handles, labels = axes[1].get_legend_handles_labels()
fig.legend(handles, labels, loc='center right', fontsize=7,
           bbox_to_anchor=(1.14, 0.5), framealpha=0.9)

plt.tight_layout(rect=[0, 0, 0.88, 0.90])
_fig_path = os.path.join(MODELS_OUT, 'errorsmith_error_rate_profiles.png')
plt.savefig(_fig_path, dpi=150, bbox_inches='tight')
_save_plot(_fig_path)
plt.show()
print("✓ Error rate profiles saved → GDrive")


# ═══════════════════════════════════════════════════════════════════════
#  PLOT 4: True vs Predicted error rate comparison (scatter)
#  Each point = one (chemistry, error_class) — perfect model = diagonal
# ═══════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(8, 8))
ax.set_title('Ground Truth vs Predicted Error Rates (corrected)\n'
             'Each point = (chemistry × error class)',
             fontsize=12, fontweight='bold')

_error_class_markers = ['s', '^', 'D', 'v']  # sub, ins, del, hp
for cls_idx in range(1, len(class_names)):  # skip class 0 (correct)
    true_rates = _true_rate_matrix[:, cls_idx]
    pred_rates = _pred_rate_matrix[:, cls_idx]
    ax.scatter(true_rates, pred_rates,
               marker=_error_class_markers[cls_idx - 1],
               color=_class_colors[cls_idx], s=60, alpha=0.8,
               label=class_names[cls_idx], edgecolors='black', linewidth=0.5)

# Perfect calibration line
_max_val = max(_true_rate_matrix[:, 1:].max(), _pred_rate_matrix[:, 1:].max()) * 1.1
ax.plot([0, _max_val], [0, _max_val], 'k--', alpha=0.4, linewidth=1, label='perfect')
ax.set_xlabel('Ground-truth rate (corrected)')
ax.set_ylabel('Predicted rate (corrected)')
ax.legend(fontsize=9)
ax.set_xlim(0, _max_val)
ax.set_ylim(0, _max_val)
ax.set_aspect('equal')
ax.grid(alpha=0.2)

plt.tight_layout()
_fig_path = os.path.join(MODELS_OUT, 'errorsmith_error_rate_calibration.png')
plt.savefig(_fig_path, dpi=150, bbox_inches='tight')
_save_plot(_fig_path)
plt.show()
print("✓ Error rate calibration plot saved → GDrive")


# ═══════════════════════════════════════════════════════════════════════
#  TABLE: Numerical summary — ground truth vs predicted error rates
#  All rates are CORRECTED for 2% correct-base subsampling.
# ═══════════════════════════════════════════════════════════════════════
_error_classes = class_names[1:]  # exclude 'correct'

print(f"\n{'=' * 90}")
print(f"Error Rate Summary — Ground Truth vs Predicted (corrected for {CORRECT_SUBSAMPLE_RATE:.0%} subsampling)")
print(f"{'=' * 90}")

hdr = f"  {'Chemistry':<28s}  {'N(csv)':>10s}  {'~N(true)':>10s}"
for ec in _error_classes:
    hdr += f"  {ec[:6]+' T':>8s}  {ec[:6]+' P':>8s}"
hdr += f"  {'TotErr T':>8s}  {'TotErr P':>8s}"
print(hdr)
print(f"  {'─'*28}  {'─'*10}  {'─'*10}" + f"  {'─'*8}  {'─'*8}" * (len(_error_classes) + 1))

for ci, cc in enumerate(_uniq_chems):
    cname = CHEMISTRY_NAMES.get(cc, f'unk_{cc}')
    counts = _chem_class_counts[cc]
    n_csv = sum(counts.values())
    # Estimated true total bases examined
    n_true_est = int(counts.get(0, 0) / CORRECT_SUBSAMPLE_RATE
                     + sum(counts.get(cls, 0) for cls in range(1, len(class_names))))
    tr = _true_rate_matrix[ci]
    pr = _pred_rate_matrix[ci]

    row = f"  {cname:<28s}  {n_csv:>10,}  {n_true_est:>10,}"
    for cls_idx, ec in enumerate(_error_classes, start=1):
        t_pct = tr[cls_idx] * 100
        p_pct = pr[cls_idx] * 100
        row += f"  {t_pct:>7.3f}%  {p_pct:>7.3f}%"
    # Total error rate (1 - correct)
    t_err = (1.0 - tr[0]) * 100
    p_err = (1.0 - pr[0]) * 100
    row += f"  {t_err:>7.3f}%  {p_err:>7.3f}%"
    print(row)

print(f"\n  T = Ground truth (corrected for correct-base subsampling)")
print(f"  P = Model prediction (corrected for correct-base subsampling)")
print(f"  N(csv) = rows in training CSV | ~N(true) = estimated true aligned bases")
print(f"  Correction: correct_count × {1/CORRECT_SUBSAMPLE_RATE:.0f} (inverse of {CORRECT_SUBSAMPLE_RATE:.0%} retention)")
print(f"  Closer T/P values = better error-rate calibration")

# ── Family assignment check ──────────────────────────────────────────
print(f"\n  ── Chemistry → Family routing check ──")
for cc in _uniq_chems:
    name = CHEMISTRY_NAMES.get(cc, f'unk_{cc}')
    fam = CHEMISTRY_TO_FAMILY.get(cc, 'UNKNOWN')
    print(f"  {name:35s} → {fam}")


  ✓ Using saved _chem_class_counts (150,035,641 rows)

  Ground-truth error rates (corrected for 2% correct-base subsampling):
  Chemistry                              Error%     Sub%     Ins%     Del%      HP%
  ───────────────────────────────────  ────────  ───────  ───────  ───────  ───────
  pacbio_hifi_sequel2                   0.270%  0.019%  0.038%  0.037%  0.176%
  ont_lsk110_r941                       7.504%  1.697%  1.283%  1.850%  2.674%
  ont_ulk001_r941                       7.172%  1.685%  1.017%  2.010%  2.460%
  ont_lsk114_r1041                      1.923%  0.651%  0.261%  0.409%  0.603%
  ont_ulk114_r1041                      4.671%  1.447%  0.905%  0.961%  1.359%
  illumina_hiseq2500                    0.956%  0.846%  0.073%  0.004%  0.033%
  pacbio_onso                           0.058%  0.056%  0.000%  0.000%  0.002%
  element_aviti                         0.087%  0.080%  0.000%  0.000%  0.007%
  element_ultraq                        0.119%  0.113%  0.000%  0.000%  0

## 7. Export Models to Google Drive

In [20]:
# ── Package and export to Google Drive ────────────────────────────────
import tarfile, shutil, glob

GDRIVE_TARBALL = os.path.join(GDRIVE_DIR, 'errorsmith_models.tar.gz')

# List all model files
model_files = glob.glob(f'{errorsmith_save_dir}/**/*', recursive=True)
print(f"Model files to export ({len(model_files)}):")
for f in sorted(model_files):
    if os.path.isfile(f):
        size_kb = os.path.getsize(f) / 1024
        print(f"  {os.path.relpath(f, errorsmith_save_dir):40s} {size_kb:>8.1f} KB")

# Create tarball
print(f"\nCreating tarball...")
with tarfile.open(GDRIVE_TARBALL, 'w:gz') as tar:
    tar.add(errorsmith_save_dir, arcname='errorsmith_models')

tarball_size = os.path.getsize(GDRIVE_TARBALL) / (1024 * 1024)
print(f"\n✅ Exported to Google Drive:")
print(f"   {GDRIVE_TARBALL}")
print(f"   Size: {tarball_size:.1f} MB")

# Also copy to Drive for easy access
if os.path.exists(GDRIVE_ERRORSMITH):
    shutil.rmtree(GDRIVE_ERRORSMITH)
shutil.copytree(errorsmith_save_dir, GDRIVE_ERRORSMITH)
print(f"   Copied → {GDRIVE_ERRORSMITH}")

print(f"\n📥 To download locally:")
print(f"   1. Open Google Drive → My Drive/Colab Notebooks/strandweaver/")
print(f"   2. Download errorsmith_models.tar.gz")
print(f"   3. Extract to strandweaver/trained_models/errorsmith/")

Model files to export (23):
  calibrators_lr_accurate.pkl                  10.4 KB
  calibrators_ont_hiacc.pkl                    11.3 KB
  calibrators_ont_lsk114_r1041.pkl             12.7 KB
  calibrators_ont_r9.pkl                       13.2 KB
  calibrators_ont_ulk114_r1041.pkl             12.9 KB
  calibrators_short_read_mixed.pkl             11.7 KB
  calibrators_short_read_sbb.pkl                2.8 KB
  error_classifier_lr_accurate.pkl          46142.3 KB
  error_classifier_lr_accurate.ubj          46138.9 KB
  error_classifier_ont_hiacc.pkl            40148.5 KB
  error_classifier_ont_hiacc.ubj            40145.1 KB
  error_classifier_ont_lsk114_r1041.pkl    107605.2 KB
  error_classifier_ont_lsk114_r1041.ubj    107601.8 KB
  error_classifier_ont_r9.pkl               77341.8 KB
  error_classifier_ont_r9.ubj               77338.4 KB
  error_classifier_ont_ulk114_r1041.pkl    133876.7 KB
  error_classifier_ont_ulk114_r1041.ubj    133873.3 KB
  error_classifier_short_read_mixed.p

## 8. Summary

In [21]:
# ── Final summary ────────────────────────────────────────────────────
ref_used = errorsmith_results.get('reference', 'unknown')
n_feat = errorsmith_results.get('n_features', len(FAMILY_FEATURES))
n_fams = errorsmith_results.get('n_families', len(family_results))

print(chr(9556) + chr(9552)*66 + chr(9559))
print(chr(9553) + "  ErrorSmith — Per-Chemistry Ensemble (v10)                       " + chr(9553))
print(chr(9568) + chr(9552)*66 + chr(9571))
print(chr(9553) + "                                                                  " + chr(9553))
print(chr(9553) + f"  {n_fams} XGBoost models, each specialised for a chemistry family:     " + chr(9553))
print(chr(9553) + "  " + chr(9472)*62 + chr(9553))

for fn in family_results:
    cv = errorsmith_results.get('families', {}).get(fn, {})
    trees = family_results[fn].get('total_trees', '?')
    f1 = cv.get('cv_f1_mean', 0)
    desc = FAMILIES[fn]['desc']
    print(chr(9553) + f"    {fn:20s}  trees={trees:>5}  F1={f1:.4f}               " + chr(9553))

print(chr(9553) + "                                                                  " + chr(9553))
print(chr(9553) + f"  Overall F1-macro: {errorsmith_results['overall_f1_macro']:.4f}"
      f"                                       " + chr(9553))
print(chr(9553) + f"  Features: {n_feat} per model (no chemistry flags!)                  " + chr(9553))
print(chr(9553) + f"  Reference: {ref_used:40s}           " + chr(9553))
print(chr(9553) + "                                                                  " + chr(9553))
print(chr(9553) + "  Output files (per family):                                      " + chr(9553))
for fn in family_results:
    print(chr(9553) + f"    " + chr(8226) + f" error_classifier_{fn}.ubj / .pkl                   " + chr(9553))
print(chr(9553) + f"    " + chr(8226) + f" family_routing.json                                    " + chr(9553))
print(chr(9553) + f"    " + chr(8226) + f" training_results.json                                  " + chr(9553))
print(chr(9553) + "                                                                  " + chr(9553))
print(chr(9553) + "  Next steps:                                                     " + chr(9553))
print(chr(9553) + "    1. Download models to strandweaver/trained_models/errorsmith/ " + chr(9553))
print(chr(9553) + "    2. Update errorsmith_module.py to load per-family models      " + chr(9553))
print(chr(9553) + "    3. Run: strandweaver core-assemble --use-ai <reads>           " + chr(9553))
print(chr(9553) + "                                                                  " + chr(9553))
print(chr(9562) + chr(9552)*66 + chr(9565))

╔══════════════════════════════════════════════════════════════════╗
║  ErrorSmith — Per-Chemistry Ensemble (v10)                       ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  7 XGBoost models, each specialised for a chemistry family:     ║
║  ──────────────────────────────────────────────────────────────║
║    ont_r9                trees=  627  F1=0.7798               ║
║    ont_lsk114_r1041      trees=  762  F1=0.8090               ║
║    ont_ulk114_r1041      trees= 1320  F1=0.8106               ║
║    ont_hiacc             trees=  682  F1=0.7788               ║
║    lr_accurate           trees= 1212  F1=0.9260               ║
║    short_read_mixed      trees= 1296  F1=0.9541               ║
║    short_read_sbb        trees= 1428  F1=0.7706               ║
║                                                                  ║
║  Overall F1-macro: 0.8552                               